# 2 — Model
Characterisation of the model: colours, UMAP (cluster / supercluster / one individual / strategies),
frame types, one individual's sequence, SeqVec families, group explanation, dendrogram / MDS,
medoids and state probabilities (cluster and supercluster).

Starts from a session saved by `1_compute.ipynb` (`LOAD_NAME` below).

In [ ]:
# ── Repository folders (RELATIVE paths: this notebook is in Code/) ──
from pathlib import Path
REPO_DIR    = Path.cwd().parent if Path.cwd().name == "Code" else Path.cwd()
DATA_DIR    = REPO_DIR / "Data"       # feature CSVs, tracking, per-individual Excel
RESULTS_DIR = REPO_DIR / "Results"    # saved sessions (e.g. reference_best) + outputs
VIDEO_ROOT  = REPO_DIR / "Videos"     # videos NOT included in the repository: put them here
_LOCAL_VIDEOS = Path("E:/01_Travail/CREC/First_Capture/VIDEO/capture")   # local copy (author's PC)
if not VIDEO_ROOT.is_dir() and _LOCAL_VIDEOS.is_dir():   # no Videos/ -> local videos
    VIDEO_ROOT = _LOCAL_VIDEOS

import json
import webbrowser
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches
from pathlib import Path
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import kneighbors_graph
import igraph as ig
import leidenalg
import umap
import pickle

%matplotlib inline
#%matplotlib qt

# ══ DATASET CHOICE ═══════════════════════════════════════════
DATASET = "cut_new"   # "cut_old" = old cut file  + old features (5)
                      # "cut_new" = new cut file  + new features (9)
                      #             (Frame_start -> strike, ~95 k frames)
                      # "full"    = full file     + new features (9)
                      #             (Start -> strike, ~1.2 M frames)

# ── CSV files
SMOOTH_FILES_CUT_OLD = {     # add_features_cut_2.py (old computation)
    1:  DATA_DIR / "cut_cuttlefish_with_features_smooth_1.csv",
   # 5:  DATA_DIR / "cut_cuttlefish_with_features_smooth_5.csv",
    #10: DATA_DIR / "cut_cuttlefish_with_features_smooth_10.csv",
    #20: DATA_DIR / "cut_cuttlefish_with_features_smooth_20.csv",
    #50: DATA_DIR / "cut_cuttlefish_with_features_smooth_50.csv",
}
SMOOTH_FILES_CUT_NEW = {     # add_features_full.py, SOURCE = "cut"
    1:  DATA_DIR / "cut_cuttlefish_with_new_features_smooth_1.csv",
   # 5:  DATA_DIR / "cut_cuttlefish_with_new_features_smooth_5.csv",
    #10: DATA_DIR / "cut_cuttlefish_with_new_features_smooth_10.csv",
    #20: DATA_DIR / "cut_cuttlefish_with_new_features_smooth_20.csv",
    #50: DATA_DIR / "cut_cuttlefish_with_new_features_smooth_50.csv",
}
SMOOTH_FILES_FULL = {        # add_features_full.py, SOURCE = "full"
    1:  DATA_DIR / "full_cuttlefish_with_features_smooth_1.csv",
}

# ── Features
FEATURES_OLD = ["distance_to_shrimp", "approach_angle", "speed_cuttle_smooth",
                "orientation_variation", "jerk_cuttle"]

# FEATURES_OLD = [
#     "distance_to_shrimp", "approach_angle", "orientation_variation",
#     "speed_cuttle_smooth", "accel_cuttle" ,"jerk_cuttle", "head_turn", "eye_convergence"
# ]

# New variables (add_features_full.py) — all except distance_change
FEATURES_NEW = [
    # 6 variables, each = one behavioural axis
    "distance_to_shrimp",        # near or far from the prey (log)
    "attack_side",               # where the cuttlefish is around the prey: 0° = in front of its head, 180° = behind
    "bearing_to_prey",           # is the cuttlefish aiming at the prey? 0° = straight ahead
    "cuttle_approach_speed",     # cuttlefish moves towards the prey (+) / backs off (-) / still (0)
    "orbit_speed",               # cuttlefish turns around the prey (°/s) -> circling
    "who_moves",                 # who moves: + = cuttlefish, - = shrimp, 0 = both / neither
    # Available in the file but removed from the clustering:
    # "speed_cuttle_log",        # redundant: approach + orbit already decompose the movement
    # "orientation_variation",   # correlated with speed (r=0.79), rotation covered by orbit_speed
    # "approach_angle",          # relative body orientation, replaced by attack_side
    # "approach_angle_variation",# half driven by the shrimp, replaced by orbit_speed
    # "speed_shrimp_log",        # prey behaviour (context, to describe afterwards)
    # "shrimp_retreat_speed",    # same
    # "distance_change",         # mixes cuttlefish + shrimp (r=0.87 with shrimp_retreat_speed)
]

# ── Feature weights (1.0 = no weighting) ────────────────────
#   Applied AFTER the StandardScaler: a weight W gives the feature W times
#   more importance in the distance between frames.
#   A feature missing from the dictionary has a weight of 1.0.
FEATURE_WEIGHTS_OLD = {      # for "cut_old" (current weights taken from cell 1)
    "orientation_variation":          8.0,
    "distance_to_shrimp":             1.0,
    "approach_angle":                 1.0,
    "speed_cuttle_smooth":            1.0,
    "jerk_cuttle":                    1.0
}

FEATURE_WEIGHTS_NEW = {      # for "cut_new" and "full"
    "distance_to_shrimp":        1.0,
    "attack_side":               3.0,
    "bearing_to_prey":           1.0,
    "cuttle_approach_speed":     2.0,   # movement: higher weight so that Leiden
    "orbit_speed":               5.0,   #   splits states by MOVEMENT rather than
    "who_moves":                 4.0,   #   by position alone
}

# ── Excluded individuals (dubious data) ──────────────────────
EXCLUDE_ANIMALS = [
    152,    # anomalous cut file (4314 frames instead of ~251) -> creates its own state
    133,    # missing from the Excel (no reliable strike frame) -> creates 2 states on its own
]

# ── Orbit threshold ───────────────────────────────────────────
#   orbit_speed is recomputed at loading (cell 2) from orbit_speed_deg_s:
#   below ORBIT_MIN_DEG_S (°/s) = slight curve, not circling -> 0, then log.
#   Without a threshold, log1p makes 3-6 °/s almost as strong as 15-50 °/s.
#   Can be changed here without rerunning add_features_full.py (0 = no threshold).
ORBIT_MIN_DEG_S = 0.0   # tested: 8 degraded the groups (single orbit state); 0 or 3 advised

# ── Leiden parameters
LEIDEN_K    = 300
LEIDEN_RES  = 1.0
LEIDEN_K2   = 3
LEIDEN_RES2 = 5.0

# ── Settings depending on DATASET ───────────────────────────
#   FRAME_STEP: Leiden + UMAP on 1 frame out of FRAME_STEP (the others: by neighbours)
FRAME_STEP = 1
VIDEO_DIR  = VIDEO_ROOT / "videos_catch"   # videoN_cut.mp4

if DATASET == "cut_old":
    SMOOTH_FILES = SMOOTH_FILES_CUT_OLD
    FEATURES     = list(FEATURES_OLD)
    OUT_DIR      = RESULTS_DIR / "analysis_results"
elif DATASET == "cut_new":
    SMOOTH_FILES = SMOOTH_FILES_CUT_NEW
    FEATURES     = list(FEATURES_NEW)
    OUT_DIR      = RESULTS_DIR / "analysis_results_cut_new"
    LEIDEN_K     = 300   # K=300 / res=0.5: more homogeneous clusters than with K=1000
    LEIDEN_RES   = 1.0   #   res 1.0: states that separate the orbit better (user choice)
elif DATASET == "full":
    SMOOTH_FILES = SMOOTH_FILES_FULL
    FEATURES     = list(FEATURES_NEW)
    OUT_DIR      = RESULTS_DIR / "analysis_results_full"
    VIDEO_DIR    = VIDEO_ROOT / "01_VIDEO_Cropped"   # videoN.mp4
    FRAME_STEP   = 3     # ~400 k frames for Leiden instead of 1.2 M
    LEIDEN_K     = 200   # 1000 neighbours × 400 k frames does not fit in memory
    LEIDEN_RES   = 0.3   # K=200 / res=0.3 -> ~17 clusters (comparable to the cut)
                         # if MemoryError: LEIDEN_K = 100 and LEIDEN_RES = 0.1
else:
    raise ValueError(f"DATASET = {DATASET!r} unknown — 'cut_old', 'cut_new' or 'full'")

USE_FULL = DATASET == "full"          # full file?
USE_NEW  = DATASET != "cut_old"       # new features?
FEATURE_WEIGHTS = dict(FEATURE_WEIGHTS_NEW if USE_NEW else FEATURE_WEIGHTS_OLD)
_unknown = [f for f in FEATURE_WEIGHTS if f not in FEATURES]
if _unknown:
    print(f"[WARN] weights defined for unused features (ignored): {_unknown}")

OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Palettes
PALETTE = [
    "#FF4FB8","#C97E48","#8C6BB1","#9C9C9C","#F5E21E",
    "#3A8891","#E45756","#4CAF50","#1F77B4","#FF9F1C",
    "#A05195","#2CB1BC","#7F7F7F","#BCBD22","#17BECF",
    "#D62728","#9467BD","#8C564B","#E377C2","#AEC7E8",
]
SUPER_PAL = [
    "#FF4444","#44AAFF","#44FF88","#FFD700","#FF44FF",
    "#FF8844","#44FFFF","#FF44AA","#88FF44","#AA44FF",
]
# ── Semantic aliases ─────────────────────────────────
CLUSTER_PALETTE  = PALETTE    # behavioural states C0, C1, ...
STRATEGY_PALETTE = SUPER_PAL  # strategies A, B, C, ... (and Ward groups)

# Fixed colormaps — single source of truth
CMAP_CLUSTER = mpl.colors.ListedColormap(PALETTE)
NORM_CLUSTER = mpl.colors.BoundaryNorm(range(len(PALETTE) + 1), ncolors=len(PALETTE))

print(f"[OK] Config — DATASET={DATASET} | {len(FEATURES)} features | "
      f"weights ≠ 1: { {f: w for f, w in FEATURE_WEIGHTS.items() if w != 1.0} or 'none'} | {len(SMOOTH_FILES)} file(s) | "
      f"1 frame / {FRAME_STEP} for Leiden | results → {OUT_DIR}")

In [ ]:
# ── Display mode ─────────────────────────────────────────────────
#   True  : each figure opens in its OWN WINDOW, one after the other —
#           close it and the next one opens (the notebook waits for you)
#   False : figures are shown inside the notebook (classic inline mode)
#   WAIT_FOR_CLOSE (windows only):
#     True  : the notebook waits until you close each window (use with "Run All")
#     False : each cell opens its window and returns at once -> several windows
#             open at the same time (run the cells one by one, not "Run All")
WAIT_FOR_CLOSE = True
SHOW_IN_WINDOWS = True

import matplotlib.pyplot as plt
if SHOW_IN_WINDOWS:
    get_ipython().run_line_magic("matplotlib", "qt")
    _show_qt = getattr(plt, "_show_qt_original", plt.show)
    plt._show_qt_original = _show_qt          # survives a rerun of this cell
    _waiting = {"on": False}

    def _show_and_wait(*args, **kwargs):
        # plt.pause() calls plt.show() internally: while waiting, just display
        if _waiting["on"]:
            return _show_qt(block=False)
        figs = plt.get_fignums()
        if not figs:
            return
        _show_qt(block=False)
        if not WAIT_FOR_CLOSE:
            return
        print(f"[FIGURE] {len(figs)} window(s) open — close to continue")
        _waiting["on"] = True
        try:
            while any(plt.fignum_exists(n) for n in figs):
                plt.pause(0.2)
        finally:
            _waiting["on"] = False

    plt.show = _show_and_wait
else:
    if hasattr(plt, "_show_qt_original"):
        plt.show = plt._show_qt_original
    get_ipython().run_line_magic("matplotlib", "inline")
print(f"[DISPLAY] {('one window after the other' if WAIT_FOR_CLOSE else 'windows, several at once') if SHOW_IN_WINDOWS else 'inline'}")

# ── Session to use ────────────────────────────────────────────
#  Set LOAD_NAME in the next cell to the session saved by 1_compute.ipynb
#  (Results/<name>, e.g. "reference_best"). Nothing is recomputed here.

In [ ]:
# ══════════════════════════════════════════════════════════════
#  RELOADING A SAVED SESSION
#  Run after cell 0 (and the cells defining the functions you need).
#  Do NOT rerun cell 2 afterwards: it would replace the reloaded clusters with the current cache.
# ══════════════════════════════════════════════════════════════
DO_LOAD        = True          # this notebook starts from a saved session (computed by 1_compute.ipynb)
LOAD_NAME      = "reference_best"            # folder name to reload; None = the most recent session
RESTORE_CONFIG = True            # True = also restores FEATURES, weights, Leiden, exclusions, orbit threshold of the session

if not DO_LOAD:
    print("[LOAD] disabled (DO_LOAD = False): memory was not changed. "
          "To reload: DO_LOAD = True, then rerun this cell.")
else:

    import json, pickle
    from pathlib import Path
    import numpy as np
    import pandas as pd

    SESSION_ROOT = RESULTS_DIR
    sessions = sorted([d for d in SESSION_ROOT.iterdir() if d.is_dir() and (d / "config.json").exists()], key=lambda d: d.stat().st_mtime)
    if not sessions:
        raise FileNotFoundError(f"No session in {SESSION_ROOT}")
    print("Available sessions :\n")
    for s in sessions:
        c = json.load(open(s / "config.json", encoding="utf-8")) if (s / "config.json").exists() else {}
        print(f"  • {s.name:<28} {c.get('date', '')}  groups explorer={c.get('n_groups_explorer', '?')}"
              f" / optimal={c.get('n_groups_optimal', '?')}  {c.get('note', '')}")
    LOAD_DIR = SESSION_ROOT / LOAD_NAME if LOAD_NAME else sessions[-1]
    if not LOAD_DIR.exists():
        raise FileNotFoundError(f"Session not found : {LOAD_DIR}")
    print(f"\n[LOAD] → {LOAD_DIR.name}\n")
    loaded, skipped = [], []


    def _load(label, fn):
        try:
            out = fn()
            if out is False:
                skipped.append(label); print(f"  [SKIP] {label}")
            else:
                loaded.append(label); print(f"  [OK]   {label}")
        except FileNotFoundError:
            skipped.append(label); print(f"  [SKIP] {label}  (file missing)")
        except Exception as e:
            skipped.append(label); print(f"  [ERR]  {label}  ({type(e).__name__}: {e})")


    # ── Configuration ──
    cfg = json.load(open(LOAD_DIR / "config.json", encoding="utf-8")) if (LOAD_DIR / "config.json").exists() else {}
    if cfg.get("DATASET") and cfg["DATASET"] != globals().get("DATASET"):
        print(f"  ⚠️  different DATASET : session={cfg['DATASET']} | current={globals().get('DATASET')}")
    def _load_cfg():
        if not RESTORE_CONFIG or not cfg:
            return False
        for k in ("SELECTED_WINDOW", "FEATURES", "FEATURE_WEIGHTS", "EXCLUDE_ANIMALS", "ORBIT_MIN_DEG_S",
                  "LEIDEN_K", "LEIDEN_RES", "LEIDEN_K2", "LEIDEN_RES2", "FRAME_STEP"):
            if cfg.get(k) is not None:
                v = cfg[k]
                if k == "SELECTED_WINDOW":
                    v = int(v)
                globals()[k] = v
    _load("config.json (features, weights, Leiden, exclusions, orbit threshold)", _load_cfg)

    # ── Frame clusters + UMAP ──
    def _load_results():
        global results, X, Z, n_cl, n_su, df
        if (LOAD_DIR / "results.pkl").exists():
            results = pickle.load(open(LOAD_DIR / "results.pkl", "rb"))
        else:                                            # old save format
            meta = json.load(open(LOAD_DIR / "results_meta.json"))
            results = {int(w): {"df": pd.read_csv(LOAD_DIR / f"df_window_{w}.csv"),
                                "X": np.load(LOAD_DIR / f"X_window_{w}.npy"),
                                "Z": np.load(LOAD_DIR / f"Z_umap_{w}.npy"),
                                "n_clusters": m["n_cl"], "n_superclusters": m["n_su"]} for w, m in meta.items()}
        w = globals().get("SELECTED_WINDOW", list(results)[0])
        w = w if w in results else list(results)[0]
        X, Z = results[w]["X"], results[w]["Z"]
        n_cl, n_su = results[w]["n_clusters"], results[w]["n_superclusters"]
        df = results[w]["df"]
    _load("results (clusters, superclusters, UMAP)", _load_results)
    def _load_df():
        global df
        if (LOAD_DIR / "df.pkl").exists():
            df = pd.read_pickle(LOAD_DIR / "df.pkl")
        elif (LOAD_DIR / "df_main.csv").exists():
            df = pd.read_csv(LOAD_DIR / "df_main.csv")
        else:
            return False
    _load("df (current df)", _load_df)

    # ── Sequences, distances and groups ──
    def _load_pickle(name, required=False):
        def _f():
            p = LOAD_DIR / f"{name}.pkl"
            if not p.exists():
                globals().pop(name, None)            # missing from the session -> removed from memory (no mixing of sessions)
                return False
            globals()[name] = pickle.load(open(p, "rb"))
        return _f
    _load("final_state (sequences, SeqVec distance, explorer groups)", _load_pickle("final_state"))
    def _load_mapping():
        p = LOAD_DIR / "mapping_optimal.json"
        if not p.exists():
            globals().pop("mapping_optimal", None)   # missing from the session -> removed from memory (no mixing of sessions)
            return False
        globals()["mapping_optimal"] = json.load(open(p))
    _load("mapping_optimal (regrouped groups)", _load_mapping)
    for _nm in ("final_state_optimal", "history", "profiles", "explication_groupes"):
        _load(_nm, _load_pickle(_nm))
    if "explication_groupes" in globals():
        explication = explication_groupes
    def _load_d():
        global D_seqvec, D_seqvec_ids
        D_seqvec = np.load(LOAD_DIR / "D_seqvec.npy")
        D_seqvec_ids = json.load(open(LOAD_DIR / "D_seqvec_ids.json")) if (LOAD_DIR / "D_seqvec_ids.json").exists() else None
    _load("D_seqvec", _load_d)
    for _nm, _fn in (("df_pairs", "df_pairs_power.csv"), ("df_merged", "df_merged_comparative.csv")):
        _load(_nm, lambda _nm=_nm, _fn=_fn: globals().__setitem__(_nm, pd.read_csv(LOAD_DIR / _fn)))
    if "final_state" in globals():
        mapping_letters = final_state.get("mapping_letters", {})

    # ── Summary ──
    print(f"\n{'=' * 60}\nSESSION RELOADED — {LOAD_DIR.name}  ({len(loaded)} loaded | {len(skipped)} missing)")
    checks = {
        "results / df":     lambda: f"{len(results)} window(s) | df {df.shape}",
        "final_state":      lambda: f"{final_state['n_cl']} states | {final_state['n_s']} groups | {final_state.get('desc', '')}",
        "mapping_optimal":  lambda: f"groups = {sorted(set(mapping_optimal.values()))}",
        "SeqVec":           lambda: f"{final_state.get('seqvec_weights')} | knn_k={final_state.get('knn_k')}",
        "FEATURE_WEIGHTS":  lambda: f"{FEATURE_WEIGHTS}",
        "LEIDEN":           lambda: f"K={LEIDEN_K} res={LEIDEN_RES} | orbit_min={globals().get('ORBIT_MIN_DEG_S')}",
    }
    for nm, fn in checks.items():
        try:
            print(f"    ✅  {nm:<18} {fn()}")
        except Exception:
            print(f"    —   {nm:<18} not defined")
    print(f"\n  Then: run the analysis / video / explain_groups() cells directly.\n  To recompute from these settings: rerun cell 2 then the explorer.\n{'=' * 60}")

## Colours

In [ ]:
# ===============================================================
#  CELL 0 — COLOUR REGISTRY (single source of truth)
#  Run BEFORE all plots.
# ===============================================================
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# ===============================================================
#  CELL A — Extraction of the results for the selected window
#  ⚠ MUST run BEFORE CELL 0 (which needs n_clusters)
# ===============================================================
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

df              = results[SELECTED_WINDOW]["df"].copy()
X_scaled        = results[SELECTED_WINDOW]["X"]
Z               = results[SELECTED_WINDOW]["Z"]
n_clusters      = results[SELECTED_WINDOW]["n_clusters"]
n_superclusters = results[SELECTED_WINDOW]["n_superclusters"]

print(f"[DATA] smooth={SELECTED_WINDOW} | {len(df)} rows | "
      f"{n_clusters} clusters | {n_superclusters} superclusters | "
      f"Z {Z.shape}")

# Check: labels must cover 0..n-1 without gaps, otherwise the
# colours (indexed by label) and the counters get shifted.
for col, n in [("cluster", n_clusters), ("supercluster", n_superclusters)]:
    if col in df.columns:
        seen = sorted(pd.to_numeric(df[col], errors="coerce").dropna().unique().astype(int))
        missing_labels = sorted(set(range(n)) - set(seen))
        extra  = sorted(set(seen) - set(range(n)))
        status_str = "OK" if not (missing_labels or extra) else f"missing={missing_labels} out_of_range={extra}"
        print(f"  {col:<13} : {len(seen)} labels present — {status_str}")

def build_color_registry(n_clusters, n_superclusters):
    """
    Builds ONE colour table, then derives everything else from it
    (cmap, norm, hex list). The two cannot diverge:
    the cmap is literally built from the list.
    """
    # 20 well-separated colours: the 10 of tab10 first
    # (the most distinct), then the 10 light variants of tab20.
    tab10 = [mcolors.to_hex(plt.get_cmap("tab10")(i)) for i in range(10)]
    tab20 = plt.get_cmap("tab20")
    light_colors = [mcolors.to_hex(tab20(i)) for i in range(1, 20, 2)]
    BASE = tab10 + light_colors                      # 20 colours, optimal order

    def _color_list(n):
        return [BASE[i % len(BASE)] for i in range(max(1, int(n)))]

    def _cmap_norm(colors):
        cmap = mcolors.ListedColormap(colors)
        # bounds at -0.5, 0.5, 1.5… -> each integer falls at the CENTRE of its bin
        norm = mcolors.BoundaryNorm(np.arange(len(colors) + 1) - 0.5,
                                    ncolors=len(colors))
        return cmap, norm

    reg = {}
    for key, n, pfx in [("cluster", n_clusters, "C"),
                        ("supercluster", n_superclusters, "S")]:
        cols = _color_list(n)
        cmap, norm = _cmap_norm(cols)
        reg[key] = {"colors": cols, "cmap": cmap, "norm": norm, "prefix": pfx}
    return reg


COLOR_REG = build_color_registry(n_clusters, n_superclusters)

# ── Backward-compatible aliases: your old code keeps working ────────
CMAP_CLUSTER = COLOR_REG["cluster"]["cmap"]
NORM_CLUSTER = COLOR_REG["cluster"]["norm"]
CMAP_SUPER   = COLOR_REG["supercluster"]["cmap"]
NORM_SUPER   = COLOR_REG["supercluster"]["norm"]
PALETTE      = COLOR_REG["cluster"]["colors"]        # used by the viewer
SUPER_PAL    = COLOR_REG["supercluster"]["colors"]   # used by the viewer


# ── THE ONLY 4 FUNCTIONS TO USE IN PLOTS ────────────────────
def group_color(label, group_col="cluster"):
    """Hex colour of a label. Use EVERYWHERE, no exceptions."""
    d = COLOR_REG[group_col]
    return d["colors"][int(label) % len(d["colors"])]


def group_prefix(group_col="cluster"):
    return COLOR_REG[group_col]["prefix"]


def group_cmap_norm(group_col="cluster"):
    d = COLOR_REG[group_col]
    return d["cmap"], d["norm"]


def group_handles(labels, group_col="cluster", lw=3, style="line"):
    """Handles ready to pass to ax.legend(handles=...)."""
    uniq = sorted({int(x) for x in np.asarray(labels).ravel()})
    pfx  = group_prefix(group_col)
    if style == "patch":
        return [Patch(color=group_color(l, group_col), label=f"{pfx}{l}") for l in uniq]
    return [Line2D([0], [0], color=group_color(l, group_col), lw=lw, label=f"{pfx}{l}")
            for l in uniq]


def preview_colors():
    """Control swatch: look at it once to validate visually."""
    fig, axes = plt.subplots(2, 1, figsize=(12, 2.4))
    for ax, key in zip(axes, ["cluster", "supercluster"]):
        cols = COLOR_REG[key]["colors"]
        for i, c in enumerate(cols):
            ax.add_patch(mpl.patches.Rectangle((i, 0), 1, 1, color=c))
            ax.text(i + 0.5, 0.5, f"{COLOR_REG[key]['prefix']}{i}",
                    ha="center", va="center", fontsize=8,
                    color="white" if i % 20 < 10 else "black", fontweight="bold")
        ax.set_xlim(0, len(cols)); ax.set_ylim(0, 1)
        ax.set_yticks([]); ax.set_xticks([])
        ax.set_title(f"{key}  ({len(cols)} groups)", fontsize=9, loc="left")
    plt.tight_layout(); plt.show()

def build_animal_colors(animal_ids, cmap_name="hsv"):
    """
    Stable colour per individual. Sorting guarantees that an animal's
    colour does NOT depend on the order in which the IDs arrive:
    same result from one run to the next.
    """
    ids = sorted({str(a).strip() for a in animal_ids}, key=natural_key)
    cm  = plt.get_cmap(cmap_name)
    COLOR_REG["animal"] = {
        "colors": {a: mcolors.to_hex(cm(i / max(1, len(ids))))
                   for i, a in enumerate(ids)},
        "prefix": "",
    }
    return COLOR_REG["animal"]["colors"]


def animal_color(aid):
    """Colour of an individual — tolerates str/int/leading zeros."""
    return COLOR_REG.get("animal", {}).get("colors", {}) \
                    .get(str(aid).strip(), "#888888")


def text_on(bg_color):
    """
    Black or white depending on the perceived luminance of the background (ITU-R BT.601).
    Avoids unreadable white text on the light tab20 shades.
    """
    r, g, b = mcolors.to_rgb(bg_color)
    return "black" if (0.299 * r + 0.587 * g + 0.114 * b) > 0.55 else "white"


def resolve_animal_id(df, raw, id_col="animal_id"):
    """
    Input -> EXACT value present in df[id_col].
    Centralised here because used by the viewer, the trajectory AND the UMAP.
    """
    vals = df[id_col].dropna().unique()
    s    = str(raw).strip()
    for v in vals:
        if str(v).strip() == s:
            return v
    try:
        num = float(s)
        for v in vals:
            try:
                if float(str(v).strip()) == num:
                    return v
            except ValueError:
                pass
    except ValueError:
        pass
    return None


def natural_key(a):
    """
    Sorts 2 before 10 (and not the reverse), tolerates text IDs.
    The tuple avoids a TypeError on mixed types: numeric values
    come first (rank 0), the others next in alphabetical order.
    """
    s = str(a).strip()
    try:
        return (0, float(s), "")
    except ValueError:
        return (1, 0.0, s)


build_animal_colors(df["animal_id"].dropna().unique())
animal_ids = sorted(df["animal_id"].dropna().unique(), key=natural_key)
n_animals  = len(animal_ids)
animal_colors_per_point = df["animal_id"].map(animal_color).to_numpy()
print(f"[COLOURS] + {n_animals} individuals registered in the registry.")

preview_colors()
print(f"[COLOURS] {n_clusters} clusters, {n_superclusters} superclusters — "
      f"single registry installed.")

In [ ]:
df              = results[SELECTED_WINDOW]["df"].copy()
X_scaled        = results[SELECTED_WINDOW]["X"]
Z               = results[SELECTED_WINDOW]["Z"]
n_clusters      = results[SELECTED_WINDOW]["n_clusters"]
n_superclusters = results[SELECTED_WINDOW]["n_superclusters"]

animal_ids  = sorted(df["animal_id"].unique())
n_animals   = len(animal_ids)
id_to_idx   = {aid: i for i, aid in enumerate(animal_ids)}

def animal_color(aid):
    return plt.cm.hsv(id_to_idx[aid] / n_animals)

animal_colors_per_point = [animal_color(aid) for aid in df["animal_id"]]

norm_cl = NORM_CLUSTER
norm_su = mpl.colors.BoundaryNorm(range(n_superclusters + 1), ncolors=n_superclusters)

print(f"[OK] Smoothing={SELECTED_WINDOW} | {n_clusters} clusters | {n_superclusters} superclusters")
print(f"     {n_animals} individuals | {len(df)} frames")

## UMAP

In [ ]:
# ===============================================================
#  CELL — UMAP  (unified colours + legends not overwritten)
#  PREREQUISITE: CELL 0 (colour registry).
#
#  Fixes vs previous version:
#   [A] supercluster: tab20 -> registry (S1 was orange in the
#       radar and light blue here)
#   [B] animal_color comes from the registry, no longer an external source
#   [C] both legends coexist (add_artist) instead of overwriting each other
#   [D] red "end" marker, no more invisible white-on-white
#   [E] highlight_animal resolved by ID (str/int) + .to_numpy() mask
#   [F] rainbow path as a LineCollection (1 artist instead of N)
#   [G] discrete colorbar with C0/C1… labels instead of a continuous scale
#   [H] labels placed on the MEDIAN (robust to crescent-shaped clouds)
# ===============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.collections import LineCollection


def _colorbar_discrete(sc, ax, n, prefix, label):
    """
    Discrete colorbar, one bin per group, with the real names.
    A continuous colorbar on integer labels is misleading:
    it suggests an order/distance between clusters that does not exist.
    """
    cb = plt.colorbar(sc, ax=ax, ticks=np.arange(n), fraction=0.046, pad=0.02)
    cb.ax.set_yticklabels([f"{prefix}{i}" for i in range(n)], fontsize=7)
    cb.set_label(label, fontsize=8)
    cb.outline.set_linewidth(0.5)
    return cb


def _annotate_groups(ax, emb, labels, n, group_col):
    """Coloured badge at the centre of each group, readable text guaranteed."""
    pfx = group_prefix(group_col)
    for c in range(n):
        m = (np.asarray(labels) == c)
        if not m.any():
            continue
        # [H] median, not mean: the centre stays INSIDE the cloud even
        #     if it is elongated, bent or bimodal.
        cx, cy = np.median(emb[m, 0]), np.median(emb[m, 1])
        bg = group_color(c, group_col)
        ax.text(cx, cy, f"{pfx}{c}",
                fontsize=8, fontweight="bold", ha="center", va="center",
                color=text_on(bg), zorder=8,
                bbox=dict(facecolor=bg, alpha=0.88, edgecolor="black",
                          linewidth=0.5, boxstyle="round,pad=0.28"))


def plot_umap(color_by="cluster", highlight_animal=None, overlay_animal=False,
              save=True, data=None, emb=None, s=3, figsize=(11, 8.5)):
    """
    color_by         : "cluster" | "supercluster" | "animal"
    highlight_animal : animal_id to highlight (rainbow path)
    overlay_animal   : True -> clusters in the background + individuals on top

    All colours come from group_color() / animal_color():
    strictly identical to plot_cluster_analysis, plot_animal_trajectory
    and the interactive viewer.
    """
    if "group_color" not in globals():
        raise RuntimeError("Run CELL 0 first (colour registry).")

    data = df if data is None else data      # globals by default
    emb  = Z  if emb  is None else emb
    emb  = np.asarray(emb)

    if len(emb) != len(data):
        raise ValueError(f"Z ({len(emb)}) and df ({len(data)}) do not have the "
                         f"same length — one of them has been filtered.")

    fig, ax = plt.subplots(figsize=figsize)
    legends = []          # [C] stack them, do not overwrite

    # ══ MODE 1 — overlay: clusters in the background + individuals on top ═════
    if overlay_animal:
        cmap_g, norm_g = group_cmap_norm("cluster")
        ax.scatter(emb[:, 0], emb[:, 1], c=data["cluster"],
                   cmap=cmap_g, norm=norm_g, s=s, alpha=0.22, zorder=1)
        for aid in animal_ids:
            m = (data["animal_id"] == aid).to_numpy()      # [E] .to_numpy()
            if m.any():
                ax.scatter(emb[m, 0], emb[m, 1], color=animal_color(aid),
                           s=s + 2, alpha=0.75, zorder=2, linewidths=0)

        legends.append(ax.legend(
            handles=[mpatches.Patch(color=animal_color(a), label=str(a))
                     for a in animal_ids],
            fontsize=6, ncol=4, loc="upper right", title="animal_id",
            title_fontsize=7, framealpha=0.9))
        _annotate_groups(ax, emb, data["cluster"].to_numpy(), n_clusters, "cluster")
        ax.set_title(f"UMAP overlay — clusters + {n_animals} individuals "
                     f"(smooth={SELECTED_WINDOW})")

    # ══ MODE 2 — coloured by cluster or supercluster ════════════════════
    elif color_by in ("cluster", "supercluster"):
        n_g = n_clusters if color_by == "cluster" else n_superclusters
        cmap_g, norm_g = group_cmap_norm(color_by)         # [A] registry

        sc = ax.scatter(emb[:, 0], emb[:, 1], c=data[color_by],
                        cmap=cmap_g, norm=norm_g, s=s, alpha=0.6, linewidths=0)

        _annotate_groups(ax, emb, data[color_by].to_numpy(), n_g, color_by)
        _colorbar_discrete(sc, ax, n_g, group_prefix(color_by), color_by)  # [G]
        ax.set_title(f"UMAP — {n_g} {color_by}s (smooth={SELECTED_WINDOW})")

    # ══ MODE 3 — coloured by individual ═══════════════════════════════════
    elif color_by == "animal":
        cols = data["animal_id"].map(animal_color).to_numpy()   # [B]
        ax.scatter(emb[:, 0], emb[:, 1], c=cols, s=s, alpha=0.6, linewidths=0)
        legends.append(ax.legend(
            handles=[mpatches.Patch(color=animal_color(a), label=str(a))
                     for a in animal_ids],
            fontsize=6, ncol=4, loc="upper right", title="animal_id",
            title_fontsize=7, framealpha=0.9))
        ax.set_title(f"UMAP — {n_animals} individuals (smooth={SELECTED_WINDOW})")

    else:
        raise ValueError(f"color_by='{color_by}' unknown — expected "
                         f"'cluster', 'supercluster' or 'animal'.")

    # ══ HIGHLIGHTING AN INDIVIDUAL ═════════════════════════════════════
    if highlight_animal is not None:
        aid_h = resolve_animal_id(data, highlight_animal)       # [E]
        if aid_h is None:
            available = sorted(str(a) for a in data["animal_id"].dropna().unique())
            print(f"[UMAP] animal {highlight_animal!r} not found — "
                  f"no highlighting. Available : {available[:15]}…")
        else:
            # .to_numpy() is essential: emb is a NumPy array indexed by
            # POSITION, whereas a pandas Series carries an index. If data was
            # filtered without reset_index, a pandas mask would
            # silently pick the wrong rows of emb.
            m_h = (data["animal_id"] == aid_h).to_numpy()
            Z_h = emb[m_h]

            if len(Z_h) > 1:
                # [F] LineCollection: 1 artist instead of N calls to plot()
                pts  = Z_h.reshape(-1, 1, 2)
                segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
                ax.add_collection(LineCollection(
                    segs, colors=plt.cm.rainbow(np.linspace(0, 1, len(segs))),
                    linewidths=1.6, alpha=0.9, zorder=5))

                ax.scatter(Z_h[0, 0], Z_h[0, 1], marker="o", s=110,
                           facecolor="#00e676", edgecolor="black",
                           linewidth=1.4, zorder=7)
                ax.scatter(Z_h[-1, 0], Z_h[-1, 1], marker="s", s=110,
                           facecolor="#ff1744", edgecolor="black",   # [D]
                           linewidth=1.4, zorder=7)
                # point at the middle of the sequence
                i_mid = len(Z_h) // 2
                ax.scatter(Z_h[i_mid, 0], Z_h[i_mid, 1], marker="D", s=95,
                           facecolor="#ffd600", edgecolor="black",
                           linewidth=1.4, zorder=7)

                # [C] pin the legends already placed before creating
                #     a new one — otherwise ax.legend() replaces them.
                for lg in legends:
                    ax.add_artist(lg)
                ax.legend(handles=[
                    Line2D([0], [0], marker="o", color="none",
                           markerfacecolor="#00e676", markeredgecolor="black",
                           markersize=9, label="start"),
                    Line2D([0], [0], marker="s", color="none",
                           markerfacecolor="#ff1744", markeredgecolor="black",
                           markersize=9, label="end"),
                    Line2D([0], [0], marker="D", color="none",
                           markerfacecolor="#ffd600", markeredgecolor="black",
                           markersize=8, label="mid-sequence"),
                    Line2D([0], [0], color="#b070d0", lw=2,
                           label="path (time →)"),
                ], fontsize=8, loc="lower left", framealpha=0.9,
                   title=f"Animal {aid_h}", title_fontsize=8)

                ax.set_title(f"{ax.get_title()}\n→ Animal {aid_h} "
                             f"({len(Z_h)} frames)")
            else:
                print(f"[UMAP] animal {aid_h!r} : {len(Z_h)} point(s), "
                      f"path cannot be drawn.")

    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_edgecolor("#cccccc")
    ax.set_xlabel("UMAP-1", fontsize=8); ax.set_ylabel("UMAP-2", fontsize=8)

    plt.tight_layout()

    if save:
        suffix = "_overlay" if overlay_animal else f"_{color_by}"
        if highlight_animal is not None:
            suffix += f"_animal{highlight_animal}"
        plt.savefig(OUT_DIR / f"umap_smooth{SELECTED_WINDOW}{suffix}.png",
                    dpi=150, bbox_inches="tight")
    plt.show()
    return fig, ax


# ═══════════════════════════════════════════════════════════════
#  COLOUR NON-REGRESSION TEST
# ═══════════════════════════════════════════════════════════════
def check_color_consistency():
    """
    Compares the colour that EACH function would assign to the same label.
    If a line shows DIVERGENT, a function is bypassing group_color().
    """
    print("=" * 58)
    print("COLOUR CONSISTENCY ACROSS PLOTS")
    print("=" * 58)
    ok_global = True
    for gc, n in [("cluster", n_clusters), ("supercluster", n_superclusters)]:
        cmap_g, norm_g = group_cmap_norm(gc)
        print(f"\n{gc} :")
        for l in range(n):
            ref      = group_color(l, gc)                       # reference
            via_cmap = mcolors.to_hex(cmap_g(norm_g(l)))        # scatter/UMAP
            status_str     = "OK" if ref == via_cmap else "DIVERGENT"
            ok_global &= (ref == via_cmap)
            print(f"  {group_prefix(gc)}{l:<3} {ref}  cmap(norm)={via_cmap}  {status_str}")
    print("\n" + ("Everything is consistent." if ok_global
                  else "⚠ Divergence detected — check CELL 0."))
    print("=" * 58)


check_color_consistency()
print("\n[OK] plot_umap defined — usage :")
print("     plot_umap('cluster')")
print("     plot_umap('supercluster', highlight_animal=12)")
print("     plot_umap(overlay_animal=True)")

In [ ]:
plot_umap(color_by="cluster")
plot_umap(color_by="supercluster")
# plot_umap(color_by="animal")
# plot_umap(overlay_animal=True)

In [ ]:
# specific animal
HIGHLIGHT_ANIMAL = resolve_animal_id(df, 42)

plot_umap(color_by="cluster",      highlight_animal=HIGHLIGHT_ANIMAL)
plot_umap(color_by="supercluster", highlight_animal=HIGHLIGHT_ANIMAL)

In [ ]:
# ── UMAP coloured by STRATEGY (each frame takes the group of its individual) ──
_groups = mapping_optimal if "mapping_optimal" in globals() else final_state["mapping_letters"]
_lab = df["animal_id"].astype(str).map(lambda a: _groups.get(a, "MONO")).to_numpy()
_names = sorted(g for g in set(_lab) if g != "MONO")
fig, ax = plt.subplots(figsize=(11, 8.5))
m = _lab == "MONO"
ax.scatter(Z[m, 0], Z[m, 1], s=2, c="#cccccc", alpha=.3, label="MONO", rasterized=True)
for k, g in enumerate(_names):
    m = _lab == g
    ax.scatter(Z[m, 0], Z[m, 1], s=2, alpha=.5, color=STRATEGY_PALETTE[k % len(STRATEGY_PALETTE)],
               label=f"{g} ({len({a for a, v in _groups.items() if v == g})} ind.)", rasterized=True)
ax.legend(markerscale=6, fontsize=8, title="strategy")
ax.set_title("UMAP of the frames — colour = strategy of the individual")
ax.set_xticks([]); ax.set_yticks([])
plt.show()

## Frame types (cluster and supercluster)

In [ ]:
# ===============================================================
#  CELL B — Group analysis: radars, heatmap, dendrogram
#  PREREQUISITES: CELL A then CELL 0 (colour registry).
#
#  Fixes vs previous version:
#   [A] hard-coded tab10 -> group_cmap_norm() (S1 was not the
#       same colour here and in the UMAP)
#   [B] BoundaryNorm built on len(unique_g) -> colour shift
#       if a group is empty. The registry indexes by LABEL.
#   [C] darkened radar titles: the light shades were
#       unreadable on a white background
#   [D] dendrogram and heatmap: labels coloured by group
#   [E] overlaid radar (direct comparison of all groups)
#   [F] safeguards: <2 groups, missing features, constant feature
# ===============================================================

import seaborn as sns
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import dendrogram, linkage


def _darken(color_in, f=0.62):
    """
    Darkens a colour so it can be written as text on a white background.
    The light tab20 variants (#aec7e8, #ffbb78…) are unreadable
    as is; the RGB channels are multiplied by f < 1.
    """
    r, g, b = mcolors.to_rgb(color_in)
    return mcolors.to_hex((r * f, g * f, b * f))


def _color_ticks(ax, ordered_labels, group_col, axis="x"):
    """
    Colours the axis labels according to the group they refer to.
    ordered_labels : list of texts already set on the axis, e.g. ["C3","C0",…]
    """
    pfx    = group_prefix(group_col)
    getter = ax.get_xticklabels if axis == "x" else ax.get_yticklabels
    for tick in getter():
        txt = tick.get_text().strip()
        if not txt.startswith(pfx):
            continue
        try:
            g = int(txt[len(pfx):])
        except ValueError:
            continue
        tick.set_color(_darken(group_color(g, group_col), 0.75))
        tick.set_fontweight("bold")


def plot_cluster_analysis(df, features, group_col="cluster", n_cols_radar=4,
                          save=True):
    """
    For a given grouping (cluster or supercluster):
      1. Individual radars + overlaid radar
      2. Heatmap of inter-group distances (min-max normalised profiles)
      3. Hierarchical dendrogram

    All colours come from group_color(): identical to
    plot_umap, plot_animal_trajectory and the interactive viewer.
    """
    if "group_color" not in globals():
        raise RuntimeError("Run CELL 0 first (colour registry).")
    if group_col not in df.columns:
        raise KeyError(f"Column '{group_col}' missing from df.")

    # [F] keep only the features that are actually present and numeric
    feats = [f for f in features
             if f in df.columns and pd.api.types.is_numeric_dtype(df[f])]
    if not feats:
        raise ValueError("No usable numeric feature.")
    if len(feats) < len(features):
        print(f"[WARN] ignored features : {sorted(set(features) - set(feats))}")

    grp      = pd.to_numeric(df[group_col], errors="coerce")
    dat      = df.loc[grp.notna()].copy()
    dat[group_col] = grp.dropna().astype(int)

    mean_g   = dat.groupby(group_col)[feats].mean()
    counts_g = dat[group_col].value_counts().sort_index()
    unique_g = mean_g.index.to_numpy()
    pfx      = group_prefix(group_col)

    if len(unique_g) == 0:
        print("[WARN] no group.")
        return None, None

    # ── Min-max normalisation PER FEATURE, across groups ───────
    # Goal: make the radar axes comparable with each other. A feature
    # that is constant from one group to another is set to 0.5 (centre)
    # rather than 0, which would be interpreted as "minimum value".
    g_min = mean_g.min(axis=0)
    g_max = mean_g.max(axis=0)
    value_range = (g_max - g_min)
    mean_g_norm = mean_g.copy()
    for f in feats:
        mean_g_norm[f] = (0.5 if value_range[f] <= 0
                          else (mean_g[f] - g_min[f]) / value_range[f])

    # [A][B] colours: indexed by LABEL, never by position
    cols_g = {int(c): group_color(int(c), group_col) for c in unique_g}

    angles   = np.linspace(0, 2 * np.pi, len(feats), endpoint=False)
    ang_full = np.concatenate([angles, [angles[0]]])
    axis_labels  = [f.replace("_", "\n") for f in feats]

    # ══ FIGURE 1 — individual radars ══════════════════════════════════
    n_cols = min(n_cols_radar, len(unique_g))
    n_rows = int(np.ceil(len(unique_g) / n_cols))
    fig1   = plt.figure(figsize=(n_cols * 4, n_rows * 4.2))

    for i, c in enumerate(unique_g):
        c   = int(c)
        ax  = fig1.add_subplot(n_rows, n_cols, i + 1, polar=True)
        v   = mean_g_norm.loc[c].to_numpy()
        vals = np.concatenate([v, [v[0]]])
        col  = cols_g[c]

        ax.plot(ang_full, vals, linewidth=2.2, color=col, zorder=3)
        ax.fill(ang_full, vals, alpha=0.28, color=col, zorder=2)
        # Grey silhouette of the overall mean, to situate the group
        ref = mean_g_norm.mean(axis=0).to_numpy()
        ax.plot(ang_full, np.concatenate([ref, [ref[0]]]),
                lw=1, ls="--", color="#999999", alpha=0.8, zorder=1)

        ax.set_xticks(angles)
        ax.set_xticklabels(axis_labels, fontsize=7)
        ax.set_ylim(0, 1)
        ax.set_yticks([0.25, 0.5, 0.75])
        ax.set_yticklabels([], fontsize=6)
        ax.grid(color="#dddddd", lw=0.6)
        ax.set_title(f"{pfx}{c}  (n={counts_g.get(c, 0)})",
                     fontsize=11, pad=14,
                     color=_darken(col), fontweight="bold")   # [C]

    fig1.suptitle(f"Radar profiles — {group_col} (smooth={SELECTED_WINDOW})\n"
                  f"min-max per feature across groups · dashed = overall mean",
                  fontsize=12, y=1.005)
    plt.tight_layout()
    if save:
        plt.savefig(OUT_DIR / f"radar_smooth{SELECTED_WINDOW}_{group_col}.png",
                    dpi=150, bbox_inches="tight")
    plt.show()

    # ══ FIGURE 2 — [E] overlaid radar + counts ═════════════════════
    fig2 = plt.figure(figsize=(13, 5.6))
    axr  = fig2.add_subplot(1, 2, 1, polar=True)
    for c in unique_g:
        c = int(c)
        v = mean_g_norm.loc[c].to_numpy()
        axr.plot(ang_full, np.concatenate([v, [v[0]]]),
                 lw=2, color=cols_g[c], label=f"{pfx}{c}", alpha=0.9)
    axr.set_xticks(angles); axr.set_xticklabels(axis_labels, fontsize=7)
    axr.set_ylim(0, 1); axr.set_yticklabels([])
    axr.grid(color="#dddddd", lw=0.6)
    axr.set_title("Profile overlay", fontsize=11, pad=16)
    axr.legend(loc="upper right", bbox_to_anchor=(1.28, 1.10),
               fontsize=7, ncol=1 if len(unique_g) <= 10 else 2,
               frameon=True, framealpha=0.9)

    axb = fig2.add_subplot(1, 2, 2)
    labs_g = [f"{pfx}{int(c)}" for c in unique_g]
    vals_n = [counts_g.get(int(c), 0) for c in unique_g]
    axb.bar(labs_g, vals_n, color=[cols_g[int(c)] for c in unique_g],
            edgecolor="black", linewidth=0.5)
    for x, v in zip(labs_g, vals_n):
        axb.text(x, v, f"{v}", ha="center", va="bottom", fontsize=7)
    axb.set_ylabel("number of frames"); axb.set_title("Counts", fontsize=11)
    axb.tick_params(axis="x", labelrotation=45)
    _color_ticks(axb, labs_g, group_col, axis="x")           # [D]
    for sp in ("top", "right"):
        axb.spines[sp].set_visible(False)

    fig2.suptitle(f"Comparison of {group_col}s (smooth={SELECTED_WINDOW})",
                  fontsize=12)
    plt.tight_layout()
    if save:
        plt.savefig(OUT_DIR / f"overlay_smooth{SELECTED_WINDOW}_{group_col}.png",
                    dpi=150, bbox_inches="tight")
    plt.show()

    # ══ FIGURE 3 — heatmap + dendrogram ══════════════════════════════
    labels = [f"{pfx}{int(c)}" for c in unique_g]

    if len(unique_g) < 2:
        print("[INFO] fewer than 2 groups: heatmap/dendrogram cannot be computed.")
        return mean_g, mean_g_norm                              # [F]

    D_mat = squareform(pdist(mean_g_norm.to_numpy()))
    Z_lnk = linkage(mean_g_norm.to_numpy(), method="ward")

    fig3, axes = plt.subplots(1, 2, figsize=(15, 5.5))

    sns.heatmap(D_mat, cmap="viridis", ax=axes[0], square=True,
                xticklabels=labels, yticklabels=labels,
                annot=len(unique_g) <= 15, fmt=".2f",
                annot_kws={"fontsize": 6},
                cbar_kws={"label": "Euclidean distance", "shrink": 0.8})
    axes[0].set_title(f"Distance between {group_col} profiles", fontsize=11)
    _color_ticks(axes[0], labels, group_col, axis="x")        # [D]
    _color_ticks(axes[0], labels, group_col, axis="y")

    # color_threshold=0 -> disable scipy's automatic colouring
    # (which codes sub-trees, not groups) to keep ONLY our
    # colour code, carried by the leaf labels.
    dendrogram(Z_lnk, ax=axes[1], labels=labels, leaf_rotation=45,
               color_threshold=0, above_threshold_color="#999999")
    axes[1].set_title(f"Hierarchical dendrogram (Ward)", fontsize=11)
    axes[1].set_ylabel("merge distance")
    _color_ticks(axes[1], labels, group_col, axis="x")        # [D]
    for sp in ("top", "right"):
        axes[1].spines[sp].set_visible(False)

    # Reminder swatches under the leaves
    axes[1].legend(handles=[Line2D([0], [0], marker="s", color="none",
                                   markerfacecolor=cols_g[int(c)],
                                   markeredgecolor="black", markersize=7,
                                   label=f"{pfx}{int(c)}")
                            for c in unique_g],
                   fontsize=6, ncol=max(1, len(unique_g) // 8),
                   loc="upper right", framealpha=0.9, title=group_col,
                   title_fontsize=7)

    plt.suptitle(f"Structure of the {group_col}s (smooth={SELECTED_WINDOW})",
                 fontsize=12)
    plt.tight_layout()
    if save:
        plt.savefig(OUT_DIR / f"dist_dendro_smooth{SELECTED_WINDOW}_{group_col}.png",
                    dpi=150, bbox_inches="tight")
    plt.show()

    # ── Terminal summary ────────────────────────────────────────────────
    print(f"\n{group_col} — {len(unique_g)} groups, {len(feats)} features")
    closest_pair = np.unravel_index(
        np.argmin(D_mat + np.eye(len(D_mat)) * D_mat.max() * 2), D_mat.shape)
    print(f"  closest pair  : {labels[closest_pair[0]]} / "
          f"{labels[closest_pair[1]]}  (d={D_mat[closest_pair]:.3f})")
    farthest_pair = np.unravel_index(np.argmax(D_mat), D_mat.shape)
    print(f"  farthest pair : {labels[farthest_pair[0]]} / "
          f"{labels[farthest_pair[1]]}  (d={D_mat[farthest_pair]:.3f})")

    return mean_g, mean_g_norm


print("[OK] plot_cluster_analysis defined — usage :")
print("     mean_c, mean_c_n = plot_cluster_analysis(df, FEATURES, 'cluster')")
print("     mean_s, mean_s_n = plot_cluster_analysis(df, FEATURES, 'supercluster')")

In [ ]:
# plot cluster windows

import os
import seaborn as sns
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import dendrogram, linkage

# ── Call ──────────────────────────────────────────────────────

plot_cluster_analysis(df, FEATURES, "cluster")
plot_cluster_analysis(df, FEATURES, "supercluster")

# Open full-size in the Windows image viewer

## Sequence of one individual

In [ ]:
# ===============================================================
#  CELL — Trajectory of one individual
#  Cuttlefish (trail coloured by group) + SHRIMP + distance
#  Requires CELL 0 (colour registry).
# ===============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D

# ── Naming conventions of the tracking points ──────────────────────
_CUTTLE_CANDS = ["TopMantle", "MidCuttleHead", "MidMantle", "MidBody"]
_PREY_CANDS   = ["MidShrimp", "HeadShrimp", "Prey"]

PREY_COLOR  = "#ffb300"     # amber — deliberately OUTSIDE the registry
LINK_COLOR  = "#9e9e9e"


def _pick_key(cands, cols):
    """First tracking point available among the candidates."""
    xset = {c[:-2] for c in cols if c.endswith("_x")}
    for c in cands:
        if c in xset:
            return c
    return None


def _runs(labels):
    """
    Compresses a sequence of labels into homogeneous runs.
    [0,0,0,1,1,2] -> [(0,3,0), (3,5,1), (5,6,2)]
    One axvspan per run instead of one per frame.
    """
    labels = np.asarray(labels)
    if len(labels) == 0:
        return []
    chg = np.flatnonzero(np.diff(labels)) + 1
    b   = np.concatenate([[0], chg, [len(labels)]])
    return [(int(b[k]), int(b[k + 1]), int(labels[b[k]])) for k in range(len(b) - 1)]


def _valid_segments(x, y):
    """
    Builds segments ONLY between consecutive points that are both
    valid. A LineCollection cannot "skip" a NaN: without this
    filtering, an untracked frame links two distant positions with a
    ghost line that never existed.

    Returns (segments, valid_points_mask, valid_pairs_mask).
    """
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y)
    if len(x) < 2:
        return np.empty((0, 2, 2)), ok, np.zeros(0, bool)
    valid_pairs = ok[:-1] & ok[1:]
    pts    = np.column_stack([x, y]).reshape(-1, 1, 2)
    segs   = np.concatenate([pts[:-1], pts[1:]], axis=1)
    return segs[valid_pairs], ok, valid_pairs


def _resolve_aid(df, raw, id_col="animal_id"):
    """Input -> exact value present in the column (str/int/zeros)."""
    if "resolve_animal_id" in globals():
        return resolve_animal_id(df, raw, id_col)
    vals = df[id_col].dropna().unique()
    s = str(raw).strip()
    for v in vals:
        if str(v).strip() == s:
            return v
    try:
        num = float(s)
        for v in vals:
            try:
                if float(str(v).strip()) == num:
                    return v
            except ValueError:
                pass
    except ValueError:
        pass
    return None


def plot_animal_trajectory(animal_id, df, group_col="cluster", features=FEATURES,
                           point_key=None, prey_key=None,
                           show_prey=True, show_distance=True,
                           link_every=0, save=True, ax_legend_ncol=2):
    """
    Left panel  : 2D trajectory — cuttlefish coloured by group,
                  shrimp as dashed amber.
    Right panel : normalised features + cuttlefish/shrimp distance,
                  background coloured by group.

    show_prey     : draw the shrimp trajectory
    show_distance : add the cuttlefish–shrimp distance curve
    link_every    : if > 0, links cuttlefish and shrimp every N frames
                    (visualises the approach; 0 = disabled)
    """
    if "group_color" not in globals():
        raise RuntimeError("Run CELL 0 first (colour registry).")

    aid = _resolve_aid(df, animal_id)
    if aid is None:
        available = sorted(str(v) for v in df["animal_id"].dropna().unique())
        print(f"[ERROR] Animal {animal_id!r} not found. Available : {available[:15]}…")
        return
    cf = df[df["animal_id"] == aid].sort_values("frame").reset_index(drop=True)
    if cf.empty:
        print(f"[ERROR] Animal {aid!r} : no rows.")
        return
    if group_col not in cf.columns:
        print(f"[ERROR] Column '{group_col}' missing from df.")
        return

    cols = set(cf.columns)
    ckey = point_key or _pick_key(_CUTTLE_CANDS, cols)
    pkey = prey_key  or _pick_key(_PREY_CANDS,   cols)
    if ckey is None:
        print(f"[ERROR] No cuttlefish point among {_CUTTLE_CANDS}.")
        return
    if show_prey and pkey is None:
        print(f"[INFO] No shrimp point among {_PREY_CANDS} "
              f"-> prey trajectory not drawn.")

    labs = pd.to_numeric(cf[group_col], errors="coerce").fillna(0).to_numpy(int)
    x = cf[f"{ckey}_x"].to_numpy(float)
    y = cf[f"{ckey}_y"].to_numpy(float)
    pfx = group_prefix(group_col)

    has_prey = show_prey and pkey is not None
    if has_prey:
        px = cf[f"{pkey}_x"].to_numpy(float)
        py = cf[f"{pkey}_y"].to_numpy(float)
    else:
        px = py = None

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # ══ 1. TRAJECTORIES ════════════════════════════════════════════════
    # ── Cuttlefish: trail coloured by group ─────────────────────────────
    segs_c, ok_c, valid_pairs_c = _valid_segments(x, y)
    if len(segs_c) == 0:
        print(f"[ERROR] Animal {aid!r} : empty/NaN cuttlefish trajectory.")
        plt.close(fig)
        return
    cols_c = [group_color(l, group_col) for l in labs[:-1][valid_pairs_c]]
    axes[0].add_collection(LineCollection(segs_c, colors=cols_c, linewidths=2.2,
                                          capstyle="round", zorder=3))
    axes[0].scatter(x[ok_c], y[ok_c],
                    c=[group_color(l, group_col) for l in labs[ok_c]],
                    s=7, zorder=4, linewidths=0)

    i0c, i1c = np.flatnonzero(ok_c)[[0, -1]]
    axes[0].scatter(x[i0c], y[i0c], marker="o", s=130, facecolor="#00e676",
                    edgecolor="black", linewidth=1.5, zorder=8)
    axes[0].scatter(x[i1c], y[i1c], marker="s", s=130, facecolor="#ff1744",
                    edgecolor="black", linewidth=1.5, zorder=8)

    # ── Shrimp: amber, dashed, OUTSIDE the group colour code ──────
    # Deliberate choice: if the prey were coloured by cluster, the two
    # trajectories could no longer be told apart. Amber is reserved for
    # the prey throughout the notebook (same in the interactive viewer).
    coverage = np.nan
    if has_prey:
        segs_p, ok_p, _ = _valid_segments(px, py)
        coverage = 100 * ok_p.mean()
        if len(segs_p):
            axes[0].add_collection(LineCollection(
                segs_p, colors=PREY_COLOR, linewidths=1.6,
                linestyles=(0, (5, 2)), alpha=0.9, zorder=2))
            axes[0].scatter(px[ok_p], py[ok_p], s=5, c=PREY_COLOR,
                            alpha=0.55, zorder=2, linewidths=0)
            i0p, i1p = np.flatnonzero(ok_p)[[0, -1]]
            axes[0].scatter(px[i0p], py[i0p], marker="^", s=110,
                            facecolor=PREY_COLOR, edgecolor="black",
                            linewidth=1.3, zorder=7)
            axes[0].scatter(px[i1p], py[i1p], marker="v", s=110,
                            facecolor="#b26a00", edgecolor="black",
                            linewidth=1.3, zorder=7)
        else:
            print(f"[INFO] Shrimp never tracked for animal {aid}.")
            has_prey = False

    # ── Cuttlefish–shrimp links (visualise the approach) ────────────────────
    if has_prey and link_every > 0:
        k = np.arange(0, len(x), link_every)
        k = k[np.isfinite(x[k]) & np.isfinite(px[k])]
        if len(k):
            links = np.stack([np.column_stack([x[k], y[k]]),
                              np.column_stack([px[k], py[k]])], axis=1)
            axes[0].add_collection(LineCollection(
                links, colors=LINK_COLOR, linewidths=0.6, alpha=0.45, zorder=1))

    # ── Distance and frame of closest approach ──────────────────────
    dist = i_min = d_min = None
    if has_prey:
        dist = np.hypot(x - px, y - py)
        if np.isfinite(dist).any():
            i_min = int(np.nanargmin(dist))
            d_min = float(dist[i_min])
            axes[0].scatter(x[i_min], y[i_min], marker="*", s=320,
                            facecolor="#ffffff", edgecolor="black",
                            linewidth=1.4, zorder=9)

    # add_collection does NOT recompute the limits -> set them by hand,
    # on the union of both trajectories.
    xs_all = np.concatenate([x[ok_c]] + ([px[np.isfinite(px)]] if has_prey else []))
    ys_all = np.concatenate([y[ok_c]] + ([py[np.isfinite(py)]] if has_prey else []))
    value_range = max(np.ptp(xs_all), np.ptp(ys_all), 1.0)
    m = 0.06 * value_range
    axes[0].set_xlim(xs_all.min() - m, xs_all.max() + m)
    axes[0].set_ylim(ys_all.min() - m, ys_all.max() + m)
    axes[0].invert_yaxis()                       # image frame: y pointing down
    axes[0].set_aspect("equal", adjustable="datalim")
    axes[0].set_title(f"Animal {aid} — Trajectories ({group_col})", fontsize=11)
    axes[0].set_xlabel("X (px)"); axes[0].set_ylabel("Y (px)")

    # ── Legend ─────────────────────────────────────────────────────────
    h_grp = group_handles(labs[ok_c], group_col, lw=3)
    leg_g = axes[0].legend(handles=h_grp, fontsize=7.5, loc="upper left",
                           ncol=ax_legend_ncol, framealpha=0.92,
                           title=f"cuttlefish — {group_col}", title_fontsize=8)
    axes[0].add_artist(leg_g)          # otherwise the next legend() overwrites it

    h_mrk = [
        Line2D([0], [0], marker="o", color="none", markerfacecolor="#00e676",
               markeredgecolor="black", markersize=9, label="cuttlefish : start"),
        Line2D([0], [0], marker="s", color="none", markerfacecolor="#ff1744",
               markeredgecolor="black", markersize=9, label="cuttlefish : end"),
    ]
    if has_prey:
        h_mrk += [
            Line2D([0], [0], color=PREY_COLOR, lw=2, ls="--",
                   label=f"shrimp ({coverage:.0f}% tracked)"),
            Line2D([0], [0], marker="^", color="none", markerfacecolor=PREY_COLOR,
                   markeredgecolor="black", markersize=9, label="shrimp : start"),
            Line2D([0], [0], marker="v", color="none", markerfacecolor="#b26a00",
                   markeredgecolor="black", markersize=9, label="shrimp : end"),
        ]
        if d_min is not None:
            h_mrk.append(Line2D([0], [0], marker="*", color="none",
                                markerfacecolor="white", markeredgecolor="black",
                                markersize=13,
                                label=f"min dist. : {d_min:.0f} px (f{i_min})"))
    axes[0].legend(handles=h_mrk, fontsize=7.5, loc="lower right",
                   framealpha=0.92, title="markers", title_fontsize=8)

    # ══ 2. TIME SERIES ══════════════════════════════════════════
    label_runs = _runs(labs)
    for a, b, lab in label_runs:
        axes[1].axvspan(a, b, color=group_color(lab, group_col),
                        alpha=0.20, lw=0, zorder=0)

    def _minmax(v):
        """Min-max normalisation, robust to NaNs and constant series."""
        v = np.asarray(v, float)
        lo, hi = np.nanmin(v), np.nanmax(v)
        return (v - lo) / (hi - lo) if hi > lo else np.full_like(v, 0.5)

    for var in features:
        if var not in cf.columns:
            continue
        v = pd.to_numeric(cf[var], errors="coerce").to_numpy(float)
        if not np.isfinite(v).any():
            continue
        axes[1].plot(_minmax(v), label=var, alpha=0.8, lw=1.2, zorder=2)

    # Cuttlefish–shrimp distance: drawn on top, in thick black
    if show_distance and dist is not None and np.isfinite(dist).any():
        axes[1].plot(_minmax(dist), color="black", lw=2.2, zorder=5,
                     label=f"cuttlefish–shrimp distance "
                           f"[{np.nanmin(dist):.0f}–{np.nanmax(dist):.0f} px]")
        axes[1].axvline(i_min, color="black", ls=":", lw=1.2, zorder=4)
        axes[1].annotate(f"min {d_min:.0f} px", xy=(i_min, 0),
                         xytext=(i_min, -0.045), ha="center", fontsize=7,
                         color="black")

    axes[1].set_xlim(0, len(cf))
    axes[1].set_ylim(-0.08, 1.05)
    axes[1].set_title(f"Animal {aid} — min-max normalised series ({group_col})",
                      fontsize=11)
    axes[1].set_xlabel("Frame (index in the sequence)")
    axes[1].set_ylabel("normalised value")

    leg_f = axes[1].legend(fontsize=6.5, loc="upper left", framealpha=0.92,
                           ncol=2, title="series", title_fontsize=8)
    axes[1].add_artist(leg_f)
    axes[1].legend(handles=group_handles(labs[ok_c], group_col, style="patch"),
                   fontsize=7, loc="upper right", framealpha=0.92,
                   ncol=ax_legend_ncol, title=group_col, title_fontsize=8)

    sup = (f"Animal {aid} | smooth={SELECTED_WINDOW} | {group_col} | "
           f"{len(cf)} frames, {len(label_runs)} runs | cuttlefish={ckey}")
    if has_prey:
        sup += f", shrimp={pkey} ({coverage:.0f}%)"
    plt.suptitle(sup, fontsize=11)
    plt.tight_layout()

    if save:
        plt.savefig(OUT_DIR /
                    f"traj_smooth{SELECTED_WINDOW}_{group_col}_animal{aid}.png",
                    dpi=150, bbox_inches="tight")
    plt.show()

    if d_min is not None:
        print(f"[TRAJ] animal {aid} : min distance {d_min:.1f} px at frame "
              f"{cf['frame'].iloc[i_min]} (index {i_min}) — "
              f"{group_col} {pfx}{labs[i_min]}")
    return cf


print("[OK] plot_animal_trajectory defined — usage :")
print("     plot_animal_trajectory(12, df, 'cluster')")
print("     plot_animal_trajectory(12, df, 'supercluster', link_every=15)")

In [ ]:
ANIMAL_ID = animal_ids[112]-1   # ← or a specific ID such as 42
ANIMAL_ID_2 = animal_ids[117]-1

plot_animal_trajectory(ANIMAL_ID, df, group_col="supercluster")
plot_animal_trajectory(ANIMAL_ID_2, df, group_col="supercluster")

## Strategies: SeqVec families, explanation, dendrogram / MDS

In [ ]:
import io, contextlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.widgets import Button, Slider, TextBox
from collections import Counter
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import pdist, squareform
from sklearn.preprocessing import StandardScaler


# ── Sequence extraction ──────────────────────────────────
def _extract_sequences(df, cluster_col, min_run_len):
    sequences, mono_ids, multi_ids = {}, [], []
    for aid, group in df.groupby("animal_id"):
        raw = group.sort_values("frame")[cluster_col].values
        blocks, lengths = [], []
        cur, start = raw[0], 0
        for i in range(1, len(raw)):
            if raw[i] != cur:
                blocks.append(int(cur)); lengths.append(i - start)
                cur, start = raw[i], i
        blocks.append(int(cur)); lengths.append(len(raw) - start)
        fb, fl = [], []
        for b, l in zip(blocks, lengths):
            if l >= min_run_len:
                if fb and fb[-1] == b: fl[-1] += l
                else: fb.append(b); fl.append(l)
        sequences[aid] = {"seq": fb, "lengths": fl,
                          "total": len(raw)}       # real duration (frames), short segments included
        (mono_ids if len(set(fb)) <= 1 else multi_ids).append(aid)
    return sequences, mono_ids, multi_ids


# ── State similarity, alignment and dynamics ───────────────
try:
    from numba import njit
except ImportError:                       # no numba: pure Python version (slower)
    def njit(*a, **k):
        return a[0] if a and callable(a[0]) else (lambda f: f)


def _state_kernel_half(state_dist):
    """Square root of the state similarity K = exp(-d²/2σ²), σ = median(d)/2.
    v @ K½ makes the Euclidean distance "soft": two close states
    (neighbouring prototypes) count as similar."""
    d = np.asarray(state_dist, float)
    sigma = np.median(d[d > 0]) / 2 if (d > 0).any() else 1.0
    K = np.exp(-d ** 2 / (2 * sigma ** 2))
    ev, evec = np.linalg.eigh(K)
    return evec @ np.diag(np.sqrt(np.clip(ev, 0, None))) @ evec.T


@njit(cache=False)
def _align_cost(a, b, sub, gap):
    """Global alignment (Needleman-Wunsch) of two state sequences.
    Substitution = distance between prototypes (0..1), insertion/deletion = gap.
    Normalised by the length of the longest sequence."""
    n, m = len(a), len(b)
    prev = np.empty(m + 1)
    for j in range(m + 1):
        prev[j] = j * gap
    for i in range(1, n + 1):
        cur = np.empty(m + 1)
        cur[0] = i * gap
        for j in range(1, m + 1):
            c = prev[j - 1] + sub[a[i - 1], b[j - 1]]
            if prev[j] + gap < c:
                c = prev[j] + gap
            if cur[j - 1] + gap < c:
                c = cur[j - 1] + gap
            cur[j] = c
        prev = cur
    return prev[m] / max(n, m, 1)


@njit(cache=False)
def _dtw_cost(x, y, radius):
    """Multivariate DTW (time × features) with a Sakoe-Chiba band:
    compares how the features evolve while tolerating a slight time shift."""
    n, m = x.shape[0], y.shape[0]
    D = np.full((n + 1, m + 1), 1e18)
    D[0, 0] = 0.0
    for i in range(1, n + 1):
        for j in range(max(1, i - radius), min(m, i + radius) + 1):
            d = 0.0
            for k in range(x.shape[1]):
                d += (x[i - 1, k] - y[j - 1, k]) ** 2
            best = D[i - 1, j - 1]
            if D[i - 1, j] < best:
                best = D[i - 1, j]
            if D[i, j - 1] < best:
                best = D[i, j - 1]
            D[i, j] = np.sqrt(d) + best
    return D[n, m]


def _pairwise(items, fn, *args):
    n = len(items)
    M = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            M[i, j] = M[j, i] = fn(items[i], items[j], *args)
    return M


# ── SeqVec ────────────────────────────────────────────────────
#   Each individual is described by 9 families, each answers one question:
#     hist     : how much time in each state?
#     temporal : which states in the 1st vs 2nd half (approach -> before the strike)?
#     lastapp  : which states just before the strike? (last appearance of each state)
#     duration : total duration of the sequence (1 column, log)
#     align    : is the order of the states similar? (close states = cheap substitution)
#     dynamics : do the features evolve the same way? (DTW on the features, without the states)
#     speed    : mean speed of the cuttlefish (1 column, log); with duration = distance travelled
#     path     : total distance travelled by the cuttlefish (mm, 1 column, log): did it move a lot?
#     start    : distance to the prey at the START of the sequence (mm, 1 column, log): does it start close?
#   Distance: D² = Σ_f w_f · (D_f / scale_f)² -> each family weighs w_f / Σw,
#   whatever its number of columns.
def _seqvec(sequences, ids, n_cl,
            w_hist=1.0, w_temporal=1.0, w_lastapp=1.0, w_duration=0.5, w_align=1.0,
            w_dynamics=1.0, w_speed=0.5, w_path=0.5, w_start=0.5, state_similarity=True,
            state_dist=None, align_gap=0.5, series=None, speed=None, path=None, start=None):

    def feat_hist(s, l, n):
        # share of time in each state (log1p: compresses extreme dominances)
        c = Counter()
        for cl, ln in zip(s, l):
            c[cl] += ln
        raw = np.array([c.get(i, 0) / sum(l) for i in range(n)])
        return np.log1p(raw * 10) / np.log1p(10)

    def feat_temporal(s, l, n):
        # share of time in each state in the 1st half (t1), 2nd half (t2)
        # and their difference (t2 - t1 > 0: state more present before the strike)
        total, half = sum(l), sum(l) / 2
        h1, h2, cum = np.zeros(n), np.zeros(n), 0
        for cl, ln in zip(s, l):
            if cl < n:
                in_h1 = max(0, min(ln, half - cum))
                h1[cl] += in_h1
                h2[cl] += ln - in_h1
            cum += ln
        t1, t2 = h1 / (half + 1e-9), h2 / (half + 1e-9)
        return np.concatenate([t1, t2, t2 - t1])

    def feat_lastapp(s, n):
        # presence (0/1) + position of the LAST appearance of each state
        # (1.0 = it is the final state, just before the strike)
        present, position = np.zeros(n), np.full(n, 0.5)
        for i in range(n):
            if i in s:
                present[i]  = 1.0
                position[i] = (len(s) - 1 - s[::-1].index(i)) / max(len(s) - 1, 1)
        return np.concatenate([present, position])

    # ── Raw vectors, grouped by family ────────────────────
    fam_sizes = {"hist": n_cl, "temporal": 3 * n_cl, "lastapp": 2 * n_cl, "duration": 1, "speed": 1, "path": 1,
                 "start": 1}
    weights   = {"hist": w_hist, "temporal": w_temporal, "lastapp": w_lastapp,
                 "duration": w_duration, "speed": w_speed if speed else 0.0,
                 "path": w_path if path else 0.0, "start": w_start if start else 0.0}
    blocks    = {f: [] for f in fam_sizes}
    for aid in ids:
        s, l = sequences[aid]["seq"], sequences[aid]["lengths"]
        if not s:                                   # empty sequence -> zero vector
            for f, k in fam_sizes.items():
                blocks[f].append(np.zeros(k))
            continue
        blocks["hist"].append(feat_hist(s, l, n_cl))
        blocks["temporal"].append(feat_temporal(s, l, n_cl))
        blocks["lastapp"].append(feat_lastapp(s, n_cl))
        # real duration of the sequence (not only the segments kept by min_run_len)
        blocks["duration"].append(np.array([np.log(sequences[aid].get("total", sum(l)))]))   # log: 100 -> 200 f = 1000 -> 2000 f
        blocks["speed"].append(np.array([np.log1p(speed.get(aid, 0.0)) if speed else 0.0]))   # mm/s, log
        blocks["path"].append(np.array([np.log1p(path.get(aid, 0.0)) if path else 0.0]))     # mm, log
        blocks["start"].append(np.array([np.log1p(start.get(aid, 0.0)) if start else 0.0]))   # mm, log
    blocks = {f: np.array(v, dtype=float) for f, v in blocks.items()}

    # ── Close states = similar ("soft" hist and temporal) ──
    if state_similarity and state_dist is not None:
        Kh = _state_kernel_half(state_dist)[:n_cl, :n_cl]
        blocks["hist"] = blocks["hist"] @ Kh
        blocks["temporal"] = (blocks["temporal"].reshape(len(ids), 3, n_cl) @ Kh
                              ).reshape(len(ids), 3 * n_cl)

    # ── Combination: each family brought to the same scale ─
    parts, used = [], {}
    for f in fam_sizes:
        if weights[f] <= 0:
            continue
        Xf = StandardScaler().fit_transform(blocks[f])
        scale = np.sqrt(np.mean(pdist(Xf) ** 2)) if len(Xf) > 1 else 0.0
        if scale < 1e-9:     # tolerance: a constant family varies by ~1e-15 (rounding)                              # constant family: adds nothing
            continue
        parts.append(Xf * np.sqrt(weights[f]) / scale)
        used[f] = weights[f]
    X  = np.hstack(parts) if parts else np.zeros((len(ids), 0))
    D2 = squareform(pdist(X)) ** 2 if parts else np.zeros((len(ids), len(ids)))

    # ── align: alignment distance of the state sequences ──────
    if w_align > 0:
        if state_dist is None:
            raise ValueError("w_align > 0 requires state_dist (provided by show_strategy_explorer)")
        sub = np.asarray(state_dist, float)[:n_cl, :n_cl]
        sub = sub / (sub.max() + 1e-12)
        sq  = [np.asarray(sequences[a]["seq"], dtype=np.int64) for a in ids]
        Da  = _pairwise(sq, _align_cost, sub, float(align_gap))
        sc  = np.sqrt(np.mean(Da[np.triu_indices(len(ids), 1)] ** 2))
        if sc > 1e-9:
            D2 += w_align * (Da / sc) ** 2
            used["align"] = w_align

    # ── dynamics: DTW on the feature series (N_DYN points per individual) ──
    if w_dynamics > 0:
        if series is None:
            raise ValueError("w_dynamics > 0 requires series (provided by show_strategy_explorer)")
        ser = [np.ascontiguousarray(series[a], dtype=float) for a in ids]
        rad = max(1, int(0.1 * ser[0].shape[0]))          # tolerated shift: 10 % of the time
        Dd  = _pairwise(ser, _dtw_cost, rad)
        sc  = np.sqrt(np.mean(Dd[np.triu_indices(len(ids), 1)] ** 2))
        if sc > 1e-9:
            D2 += w_dynamics * (Dd / sc) ** 2
            used["dynamics"] = w_dynamics

    if not used:
        raise ValueError("All SeqVec weights are 0 (or constant families).")
    D   = np.sqrt(D2)
    tot = sum(used.values())
    print("[SEQVEC] mean share of each family in D² : " +
          " | ".join(f"{f} {w / tot:.0%}" for f, w in used.items()))
    desc = f"SeqVec ({', '.join(used)})"
    print(f"[SEQVEC] {len(ids)} individuals | {desc}")
    return X, D, desc
# ── Hierarchical clustering ───────────────────────────────────
def _cluster_hier(D, ids, n_strat, linkage_method="ward"):
    Z      = linkage(squareform(np.clip(D, 0, None)), method=linkage_method)
    labels = fcluster(Z, n_strat, criterion="maxclust")
    n2l    = {n: str(i) for i, n in enumerate(sorted(set(labels)))}
    return {aid: n2l[lab] for aid, lab in zip(ids, labels)}, Z


def _ward_stability(D, k, linkage_method="ward", n_boot=30, seed=0):
    """Group stability: remove 20 % of the individuals, redo the
    clustering and measure the agreement (ARI) with the full clustering."""
    from sklearn.metrics import adjusted_rand_score
    cut = lambda M: fcluster(linkage(squareform(np.clip(M, 0, None), checks=False),
                                     method=linkage_method), k, criterion="maxclust")
    rng, full, aris = np.random.default_rng(seed), cut(D), []
    for _ in range(n_boot):
        sub = np.sort(rng.choice(len(D), int(.8 * len(D)), replace=False))
        aris.append(adjusted_rand_score(full[sub], cut(D[np.ix_(sub, sub)])))
    return float(np.mean(aris))


def _silhouette_optimal_k(D, k_range=range(2, 13), linkage_method="ward"):
    """K that maximises the silhouette of the hierarchical clustering on D.
    Silhouette: is each individual closer to its own group than to the
    neighbouring group? (1 = clean groups, 0 = overlap)."""
    from sklearn.metrics import silhouette_score
    Zl  = linkage(squareform(np.clip(D, 0, None), checks=False), method=linkage_method)
    res = {}
    print("[SILHOUETTE] K tested (Ward) :")
    for k in k_range:
        if k >= len(D):
            break
        lab = fcluster(Zl, k, criterion="maxclust")
        if len(set(lab)) < 2:
            continue
        res[k] = silhouette_score(D, lab, metric="precomputed")
        print("   K={k:<3} silhouette {s:+.3f} | smallest group {m}".format(k=k, s=res[k], m=int(np.bincount(lab)[1:].min())))
    k_best = max(res, key=res.get)
    print(f"[SILHOUETTE] K selected : {k_best}")
    return k_best, res


# ── Leiden to estimate the optimal K ────────────────────────────
def _leiden_optimal_k(D, ids, resolution_range=None):
    try:
        import igraph as ig
        import leidenalg
    except ImportError:
        print("[WARN] igraph/leidenalg not available")
        return None, {}
    n = len(ids)
    if n < 3: return None, {}
    D_max = D.max()
    if D_max == 0: return 1, {}
    sim       = 1 - D / D_max
    threshold = np.percentile(sim[sim > 0], 50)
    edges, weights = [], []
    for i in range(n):
        for j in range(i+1, n):
            if sim[i, j] > threshold:
                edges.append((i, j)); weights.append(float(sim[i, j]))
    if not edges: return 1, {}
    g = ig.Graph(n=n, edges=edges)
    g.es["weight"] = weights
    if resolution_range is None:
        resolution_range = [0.5, 0.8, 0.9, 1.0, 1.1, 1.2, 1.5, 2.0]
    results = {}
    for res in resolution_range:
        part = leidenalg.find_partition(
            g, leidenalg.RBConfigurationVertexPartition,
            weights="weight", resolution_parameter=res, seed=42)
        # standard modularity (comparable across resolutions, unlike
        # part.quality() which drops mechanically as the resolution rises)
        results[res] = {"k": len(set(part.membership)),
                        "modularity": g.modularity(part.membership, weights="weight")}
    ok = {r: v for r, v in results.items() if v["k"] >= 2}   # at least 2 groups
    if not ok: return 1, results
    best_res  = max(ok, key=lambda r: ok[r]["modularity"])
    k_optimal = results[best_res]["k"]
    print(f"\n[LEIDEN] optimal K = {k_optimal} (resolution={best_res:.2f})")
    for res, r in results.items():
        print(f"  res={res:.2f} → K={r['k']} modularity={r['modularity']:.3f}")
    return k_optimal, results


def _leiden_knn_groups(D, ids, knn_k=10, min_size=3,
                       resolutions=(0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.5)):
    """Groups = Leiden communities on the graph of the knn_k nearest neighbours.
    knn_k = number of NEIGHBOURS per individual (NOT the number of groups):
    Leiden decides the number of groups itself (resolution with the best modularity).
    Groups with < min_size individuals are attached to the closest group.
    Tested (test_k_methods.py): best method on synthetic and real data."""
    import igraph as ig
    import leidenalg
    n   = len(ids)
    sim = 1 - D / (D.max() + 1e-12)
    nn  = np.argsort(D, axis=1)[:, 1:knn_k + 1]
    edges = sorted({(min(i, j), max(i, j)) for i in range(n) for j in nn[i]})
    w = [float(sim[i, j]) for i, j in edges]
    g = ig.Graph(n=n, edges=edges)
    best, best_q, best_res = None, -np.inf, None
    for res in resolutions:
        p = leidenalg.find_partition(g, leidenalg.RBConfigurationVertexPartition,
                                     weights=w, resolution_parameter=res, seed=42)
        mem = np.array(p.membership)
        if len(set(mem)) < 2:
            continue
        q = g.modularity(mem, weights=w)
        if q > best_q:
            best, best_q, best_res = mem, q, res
    if best is None:
        return {a: "0" for a in ids}, 1
    lab = best.copy()
    while True:                                  # attach the small groups
        sizes = pd.Series(lab).value_counts()
        small, big = sizes[sizes < min_size].index, sizes[sizes >= min_size].index
        if len(small) == 0 or len(big) == 0:
            break
        for i in np.where(np.isin(lab, small))[0]:
            lab[i] = min(big, key=lambda c: D[i, lab == c].mean())
    order = pd.Series(lab).value_counts().index          # 0 = largest group
    l2s   = {c: str(r) for r, c in enumerate(order)}
    print(f"[LEIDEN kNN] {knn_k} neighbours | resolution {best_res} | modularity {best_q:.3f} "
          f"| {len(order)} groups")
    return {a: l2s[c] for a, c in zip(ids, lab)}, len(order)


def _knn_stability(D, knn_k=10, n_boot=30, seed=0):
    """Stability: remove 20 % of the individuals, redo the kNN groups and
    measure the agreement (ARI) with the full groups (> 0.75 good, < 0.5 fragile)."""
    from sklearn.metrics import adjusted_rand_score
    ids = list(range(len(D)))
    with contextlib.redirect_stdout(io.StringIO()):
        full = _leiden_knn_groups(D, ids, knn_k)[0]
        rng, aris = np.random.default_rng(seed), []
        for _ in range(n_boot):
            sub = np.sort(rng.choice(len(D), int(.8 * len(D)), replace=False))
            part = _leiden_knn_groups(D[np.ix_(sub, sub)], list(sub), knn_k)[0]
            aris.append(adjusted_rand_score([full[i] for i in sub], [part[i] for i in sub]))
    return float(np.mean(aris))


# ── Main window ────────────────────────────────────────
def show_strategy_explorer(
    df,
    cluster_col    = "cluster",
    n_strategies   = 4,
    min_run_len      = 3,
    use_leiden_k   = False,
    k_method       = None,            # "knn" (recommended) | "leiden" | "silhouette" | a number
    knn_k          = 10,              # k_method='knn': number of NEIGHBOURS per individual (not the number of groups)
    k_range        = range(2, 13),    # K tested by k_method='silhouette'
    linkage_method = "ward",
    # ── SeqVec: weights of the 9 families (see _seqvec) ──
    w_hist     = 1.0,     # time spent in each state
    w_temporal = 1.0,     # states at the start vs end of the sequence
    w_lastapp  = 1.0,     # states just before the strike
    w_duration = 0.5,     # total duration (1 column)
    w_align    = 1.0,     # order of the states (alignment)
    w_dynamics = 1.0,     # evolution of the features (DTW), without the states
    w_speed    = 0.5,     # mean speed (1 column); with duration = distance travelled
    w_path     = 0.5,     # total distance travelled (1 column)
    w_start    = 0.5,     # start distance to the prey (1 column): separates close-range strikes
    state_similarity = True,   # close states = similar
    features   = None,
):
    df = df.copy()
    df["animal_id"] = df["animal_id"].astype(str)

    # ── SeqVec inputs: state prototypes + feature series ──
    feats  = list(features) if features is not None else list(FEATURES)
    fw     = globals().get("FEATURE_WEIGHTS", {}) or {}
    zs_all = pd.DataFrame(((df[feats] - df[feats].mean()) / df[feats].std()).values
                          * np.array([fw.get(f, 1.0) for f in feats]), index=df.index)
    seqvec_inputs = {}
    N_DYN = 50          # points per individual for the DTW (duration is in the duration family)

    def _seqvec_inputs(col, n_cl):
        # state prototype = centroid in the standardised + weighted space
        cent = zs_all.groupby(df[col].values).mean().reindex(range(n_cl)).fillna(0).values
        # series of each individual: N_DYN successive means (normalised time)
        ser = {}
        for aid, g in df.groupby("animal_id"):
            v = zs_all.loc[g.sort_values("frame").index].values
            e = np.linspace(0, len(v), N_DYN + 1).astype(int)
            ser[aid] = np.vstack([v[e[i]:max(e[i + 1], e[i] + 1)].mean(0) for i in range(N_DYN)])
        # mean speed of each individual (speed family)
        spd = (df.groupby("animal_id")["speed_cuttle_mm_s"].mean().to_dict()
               if "speed_cuttle_mm_s" in df.columns else None)
        # total distance travelled by each individual (path family, mm)
        pth = ((df.groupby("animal_id")["speed_cuttle_mm_s"].sum() / 60).to_dict()
               if "speed_cuttle_mm_s" in df.columns else None)
        # distance to the prey at the start of the sequence (start family, mm)
        st0 = (df.sort_values("frame").groupby("animal_id")["distance_mm"].first().to_dict()
               if "distance_mm" in df.columns else None)
        return {"state_dist": squareform(pdist(cent)), "state_similarity": state_similarity,
                "series": ser,
                "speed": spd,
                "path": pth,
                "start": st0}

    def compute_all(col, n_s):
        n_cl = int(df[col].nunique())
        seqs, mono, multi = _extract_sequences(df, col, min_run_len)
        print(f"[INFO] {len(multi)} multi | {len(mono)} mono | n_cl={n_cl}")
        if len(multi) < 2:
            return seqs, mono, multi, None, None, {}, n_cl, "—", n_s
        seqvec_inputs.clear()
        seqvec_inputs.update(_seqvec_inputs(col, n_cl))
        X, D, desc = _seqvec(
            seqs, multi, n_cl,
            w_hist=w_hist, w_temporal=w_temporal, w_lastapp=w_lastapp,
            w_duration=w_duration, w_align=w_align, w_dynamics=w_dynamics,
            w_speed=w_speed, w_path=w_path, w_start=w_start, **seqvec_inputs,
        )
        method = (k_method if k_method is not None
                  else ("leiden" if use_leiden_k else n_s))
        if method == "knn":
            strat, n_s = _leiden_knn_groups(D, multi, knn_k)
            Z    = linkage(squareform(np.clip(D, 0, None), checks=False), method=linkage_method)  # dendrogram
            stab = _knn_stability(D, knn_k)
            print(f"[STABILITY] K={n_s} : bootstrap ARI {stab:.2f} (> 0.75 good, < 0.5 fragile)")
            print(f"[CLUSTER] {Counter(strat.values())} | {desc}")
            return seqs, mono, multi, D, Z, strat, n_cl, desc, n_s
        if method == "leiden":
            k_opt, _ = _leiden_optimal_k(D, multi)
            if k_opt is not None:
                n_s = k_opt
                print(f"[LEIDEN] K selected : {n_s}")
        elif method == "silhouette":
            n_s, _ = _silhouette_optimal_k(D, k_range, linkage_method)
        else:
            try:
                n_s = int(method)
            except (TypeError, ValueError):
                raise ValueError(f"k_method must be 'knn', 'leiden', 'silhouette' or a number (got {k_method!r})")
            if k_method is not None:
                print(f"[K] set by hand : {n_s}")
        strat, Z = _cluster_hier(D, multi, n_s, linkage_method)
        stab = _ward_stability(D, n_s, linkage_method)
        print(f"[STABILITY] K={n_s} : bootstrap ARI {stab:.2f} (> 0.75 good, < 0.5 fragile)")
        print(f"[CLUSTER] {Counter(strat.values())} | {desc}")
        return seqs, mono, multi, D, Z, strat, n_cl, desc, n_s

    seqs, mono, multi, D, Z, strat, n_cl, desc, n_s_eff = compute_all(
        cluster_col, n_strategies)

    cur = {"col": cluster_col, "n_s": n_s_eff,
           "seqs": seqs, "mono": mono, "multi": multi,
           "D": D, "Z": Z, "strat": strat, "n_cl": n_cl, "desc": desc}

    # ── Figure ────────────────────────────────────────────────
    fig = plt.figure(figsize=(24, 12))
    fig.patch.set_facecolor("#1a1a2e")

    ax_dend = fig.add_axes([0.02, 0.22, 0.19, 0.72])
    ax_heat = fig.add_axes([0.23, 0.22, 0.19, 0.72])
    ax_seq  = fig.add_axes([0.45, 0.60, 0.53, 0.34])
    zone_r  = [0.45, 0.22, 0.53, 0.34]

    for ax in [ax_dend, ax_heat, ax_seq]:
        ax.set_facecolor("#0d0d1a")
        for sp in ax.spines.values(): sp.set_edgecolor("#333333")
        ax.tick_params(colors="#555555", labelsize=7)

    ax_s_k   = fig.add_axes([0.02, 0.11, 0.19, 0.03])
    ax_info  = fig.add_axes([0.02, 0.03, 0.60, 0.05])
    ax_tb    = fig.add_axes([0.66, 0.11, 0.14, 0.04])
    ax_bshow = fig.add_axes([0.81, 0.11, 0.08, 0.04])
    ax_bleid = fig.add_axes([0.66, 0.04, 0.23, 0.04])

    slider   = Slider(ax_s_k, "K groups", 2,
                      max(3, min(12, len(multi)-1) if len(multi) > 2 else 12),
                      valinit=n_s_eff, valstep=1, color="#4466aa")
    textbox  = TextBox(ax_tb,    "Animal ID: ", initial="",
                       color="#111122", hovercolor="#1a1a2e")
    btn_show = Button(ax_bshow, "Show",  color="#224422", hovercolor="#336633")
    btn_leid = Button(ax_bleid, "Leiden kNN groups",
                      color="#442244", hovercolor="#663366")

    for btn in [btn_show, btn_leid]:
        btn.label.set_color("#cccccc"); btn.label.set_fontsize(9)
    slider.label.set_color("white"); slider.valtext.set_color("white")
    textbox.label.set_color("#aaaaaa"); textbox.text_disp.set_color("#ffffff")

    ax_info.axis("off")
    info_txt = ax_info.text(
        0, 0.5,
        f"col={cluster_col} | seqvec | min_run_len={min_run_len} | "
        f"K={n_s_eff} | {len(multi)} multi + {len(mono)} MONO",
        color="#888888", fontsize=8, va="center",
        transform=ax_info.transAxes, fontfamily="monospace")

    radar_axes = []

    def remove_radars():
        for ax in radar_axes: ax.remove()
        radar_axes.clear()

    def update_info():
        info_txt.set_text(
            f"col={cur['col']} | seqvec | min_run_len={min_run_len} | "
            f"K={cur['n_s']} | {len(cur['multi'])} multi + {len(cur['mono'])} MONO")

    def draw():
        seqs  = cur["seqs"]; mono  = cur["mono"]; multi = cur["multi"]
        D     = cur["D"];    Z     = cur["Z"];    strat = cur["strat"]
        n_cl  = cur["n_cl"]; n_s   = cur["n_s"]

        strategy_list = sorted(set(strat.values())) if strat else []
        pal = {s: STRATEGY_PALETTE[i % len(STRATEGY_PALETTE)]
               for i, s in enumerate(strategy_list)}
        pal["MONO"] = "#666666"

        # ── Dendrogram ──────────────────────────────────────
        ax_dend.cla(); ax_dend.set_facecolor("#0d0d1a")
        for sp in ax_dend.spines.values(): sp.set_edgecolor("#333333")
        ax_dend.tick_params(colors="#555555", labelsize=6)
        if Z is not None and len(multi) > 2:
            cut_h = Z[-(n_s-1), 2]*1.01 if len(Z) >= n_s else Z[-1, 2]
            dendrogram(Z, ax=ax_dend, no_labels=True,
                       color_threshold=cut_h,
                       above_threshold_color="#444455")
            ax_dend.axhline(cut_h, color="#FFD700", lw=1.2, ls="--",
                            label=f"K={n_s}")
            ax_dend.legend(fontsize=7, labelcolor="white",
                           loc="upper right").get_frame().set_facecolor("#111111")
        ax_dend.set_title(
            f"Dendrogram\n{len(multi)} multi + {len(mono)} MONO",
            color="#aaaaaa", fontsize=8, pad=6)
        ax_dend.set_xlabel("Individuals", color="#555555", fontsize=7)
        ax_dend.set_ylabel("Distance",  color="#555555", fontsize=7)

        # ── Heatmap ───────────────────────────────────────────
        ax_heat.cla(); ax_heat.set_facecolor("#0d0d1a")
        for sp in ax_heat.spines.values(): sp.set_edgecolor("#333333")
        if D is not None and len(multi) > 0:
            sorted_ids = sorted(multi, key=lambda x: strat.get(x, "?"))
            idx_map    = {aid: i for i, aid in enumerate(multi)}
            order      = [idx_map[aid] for aid in sorted_ids]
            Ds         = D[np.ix_(order, order)]
            im = ax_heat.imshow(Ds, cmap="viridis", aspect="auto")
            ax_heat.set_xticks([]); ax_heat.set_yticks([])
            prev_s, xp = None, 0
            for aid in sorted_ids:
                s = strat.get(aid, "?")
                if s != prev_s and prev_s is not None:
                    ax_heat.axhline(xp-.5, color="white", lw=0.8, alpha=0.5)
                    ax_heat.axvline(xp-.5, color="white", lw=0.8, alpha=0.5)
                prev_s = s; xp += 1
            plt.colorbar(im, ax=ax_heat, fraction=0.03, pad=0.02,
                         label="Dist.").ax.tick_params(colors="white", labelsize=5)
        patches = ([mpatches.Patch(color=pal[s],
                    label=f"{s} (n={sum(1 for v in strat.values() if v==s)})")
                    for s in strategy_list]
                   + [mpatches.Patch(color="#666666",
                      label=f"MONO (n={len(mono)})")])
        ax_heat.legend(handles=patches, fontsize=6, loc="lower right",
                       framealpha=0.5).get_frame().set_facecolor("#111111")
        ax_heat.set_title(f"Distances\n{cur['desc'][:35]}",
                          color="#aaaaaa", fontsize=8, pad=6)

        # ── Radars ────────────────────────────────────────────
        remove_radars()
        all_s = strategy_list + (["MONO"] if mono else [])
        nc = min(5, len(all_s)); nr = (len(all_s)+nc-1)//nc
        w_r = zone_r[2]/nc; h_r = zone_r[3]/nr

        for si, s in enumerate(all_s):
            members = mono if s == "MONO" else \
                      [aid for aid, lab in strat.items() if lab == s]
            profs = []
            for aid in members:
                sq = seqs[aid]["seq"]; lng = seqs[aid]["lengths"]
                total = sum(lng)
                if total == 0: continue
                h = np.zeros(n_cl)
                for cl, l in zip(sq, lng):
                    if cl < n_cl: h[cl] += l / total
                profs.append(h)
            if not profs: continue
            mp = np.mean(profs, axis=0)
            ci = si % nc; ri = si // nc
            rect = [zone_r[0]+ci*w_r, zone_r[1]+(nr-1-ri)*h_r,
                    w_r*0.90, h_r*0.88]
            ax_r = fig.add_axes(rect, polar=True)
            ax_r.set_facecolor("#0d0d1a"); radar_axes.append(ax_r)
            angs = np.linspace(0, 2*np.pi, n_cl, endpoint=False)
            af   = np.concatenate([angs, [angs[0]]])
            vf   = np.concatenate([mp, [mp[0]]])
            col  = pal.get(s, "#888888")
            ax_r.plot(af, vf, lw=2, color=col)
            ax_r.fill(af, vf, alpha=0.3, color=col)
            ax_r.set_xticks(angs)
            ax_r.set_xticklabels([f"C{c}" for c in range(n_cl)],
                                  fontsize=4, color="#aaaaaa")
            ax_r.set_ylim(0, max(mp.max()*1.2, 0.01))
            ax_r.set_yticks([])
            ax_r.grid(color="#333333", lw=0.5)
            ax_r.spines["polar"].set_color("#333333")
            ax_r.set_title(f"{s} n={len(members)}",
                           color=col, fontsize=7, pad=4)

        fig.suptitle(
            f"Strategies | col={cur['col']} | SeqVec+ | "
            f"K={n_s} | {len(multi)} multi + {len(mono)} MONO",
            color="#cccccc", fontsize=10, y=0.99)
        update_info()
        fig.canvas.draw_idle()

    draw()

    # ── Individual timeline ─────────────────────────────────────
    def show_indiv(event=None):
        raw   = textbox.text.strip()
        seqs  = cur["seqs"]; strat = cur["strat"]
        strategy_list = sorted(set(strat.values())) if strat else []
        pal = {s: STRATEGY_PALETTE[i % len(STRATEGY_PALETTE)]
               for i, s in enumerate(strategy_list)}
        pal["MONO"] = "#666666"

        aid = str(raw) if raw in seqs else None
        if aid is None and raw.isdigit():
            aid_int = str(int(raw))
            if aid_int in seqs: aid = aid_int

        ax_seq.cla(); ax_seq.set_facecolor("#0d0d1a")
        for sp in ax_seq.spines.values(): sp.set_edgecolor("#333333")

        if aid is None:
            sample = list(seqs.keys())[:8]
            ax_seq.text(0.5, 0.5,
                f"'{raw}' not found\nExamples : {sample}",
                ha="center", va="center", color="#ff4444",
                transform=ax_seq.transAxes, fontsize=9)
            fig.canvas.draw_idle(); return

        sq  = seqs[aid]["seq"]; lng = seqs[aid]["lengths"]
        s   = strat.get(aid, "MONO"); col = pal.get(s, "#888888")
        ax_seq.set_title(
            f"Animal {aid}  —  Strategy {s}  |  "
            f"{len(sq)} blocks, {sum(lng)} frames",
            color=col, fontsize=9)
        ax_seq.set_yticks([])
        ax_seq.tick_params(colors="#555555", labelsize=7)

        x = 0
        for cluster, length in zip(sq, lng):
            c = PALETTE[cluster % len(PALETTE)]
            ax_seq.barh(0, length, left=x, height=0.6,
                        color=c, edgecolor="#111122", lw=0.4)
            if length >= max(lng) * 0.06:
                ax_seq.text(x + length/2, 0, str(cluster),
                            ha="center", va="center",
                            fontsize=7, color="white", fontweight="bold")
            x += length

        ax_seq.set_xlim(0, x); ax_seq.set_ylim(-0.5, 0.5)
        ax_seq.set_xlabel("Frames", color="#555555", fontsize=7)
        ax_seq.grid(axis="x", alpha=0.15, color="#334455")
        patches = [mpatches.Patch(color=PALETTE[c % len(PALETTE)],
                   label=f"C{c}") for c in sorted(set(sq))]
        ax_seq.legend(handles=patches, fontsize=6, loc="upper right",
                      framealpha=0.5).get_frame().set_facecolor("#111111")
        fig.canvas.draw_idle()

    # ── Events ────────────────────────────────────────────
    def on_leiden(event):
        if cur["D"] is None: return
        print("[LEIDEN kNN] Computing...")
        strat, k_opt = _leiden_knn_groups(cur["D"], cur["multi"], knn_k)
        cur["n_s"] = k_opt
        slider.eventson = False; slider.set_val(min(k_opt, slider.valmax)); slider.eventson = True
        cur.update({"strat": strat})
        remove_radars(); draw()

    def on_slider(val):
        cur["n_s"] = int(val)
        if cur["D"] is not None:
            strat, Z = _cluster_hier(cur["D"], cur["multi"],
                                      cur["n_s"], linkage_method)
            cur.update({"strat": strat, "Z": Z})
        remove_radars(); draw()

    slider.on_changed(on_slider)
    btn_show.on_clicked(show_indiv)
    btn_leid.on_clicked(on_leiden)
    textbox.on_submit(lambda t: show_indiv())

    plt.show()

    # ── Return final_state ─────────────────────────────────
    # ── Return final_state ─────────────────────────────────
    strat_final = cur["strat"] if cur["strat"] else {}
    mono_final  = cur["mono"]

    # Convert numbers → letters (A, B, C...)
    unique_groups   = sorted(set(v for v in strat_final.values()))
    num_to_letter   = {g: chr(65+i) for i, g in enumerate(unique_groups)}
    mapping_letters = {aid: num_to_letter.get(g, "MONO")
                       for aid, g in strat_final.items()}
    for m in mono_final:
        mapping_letters[str(m)] = "MONO"

    return {
        "seqs"           : cur["seqs"],
        "strat"          : strat_final,
        "mono"           : mono_final,
        "multi"          : cur["multi"],
        "n_cl"           : cur["n_cl"],
        "col"            : cur["col"],   # ← added
        "D"              : cur["D"],
        "Z"              : cur["Z"],
        "n_s"            : cur["n_s"],
        "desc"           : cur["desc"],
        "mapping_letters": mapping_letters,
        "seqvec_inputs"  : seqvec_inputs,
        "seqvec_weights" : dict(w_hist=w_hist, w_temporal=w_temporal, w_lastapp=w_lastapp,
                                w_duration=w_duration, w_align=w_align, w_dynamics=w_dynamics,
                                w_speed=w_speed, w_path=w_path, w_start=w_start),
        "knn_k"          : knn_k,
        "min_run_len"      : min_run_len,
        }

In [ ]:
# ===============================================================
#  CELL — Diagnostics of the SeqVec families
#  Tests each family ALONE, then combinations, to see
#  which one separates the groups and which one adds noise.
# ===============================================================
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics import silhouette_score


def diagnose_seqvec_families(final_state, k_range=(3, 4, 5, 6),
                             linkage_method="ward"):
    """
    For each isolated SeqVec family (weight 1, the others at 0), rebuilds
    D, clusters, and measures the best silhouette over k_range.
    _seqvec must be defined in the notebook.
    """
    seqs  = final_state["seqs"]
    multi = final_state["multi"]
    n_cl  = final_state["n_cl"]

    families = ["hist", "temporal", "lastapp", "duration", "align", "dynamics", "speed", "path", "start"]

    def best_sil(D):
        Zl = linkage(squareform(np.clip(D, 0, None)), method=linkage_method)
        best = -1
        for k in k_range:
            lab = fcluster(Zl, k, criterion="maxclust")
            if len(set(lab)) < 2:
                continue
            s = silhouette_score(D, lab, metric="precomputed")
            best = max(best, s)
        return best

    print("=" * 58)
    print("SILHOUETTE PER SEQVEC FAMILY (isolated)")
    print("=" * 58)
    print(f"  {'family':<12}{'silhouette':>12}  verdict")
    print("  " + "-" * 40)

    scores = {}
    for fam in families:
        w = {f"w_{x}": (1.0 if x == fam else 0.0) for x in families}
        try:
            _, D, _ = _seqvec(seqs, multi, n_cl, **w,
                        **final_state.get("seqvec_inputs", {}))
            s = best_sil(D)
            scores[fam] = s
            verdict = ("SEPARATES well" if s > 0.25 else
                       "a little" if s > 0.10 else "noise")
            print(f"  {fam:<12}{s:>+12.3f}  {verdict}")
        except Exception as e:
            print(f"  {fam:<12}{'ERROR':>12}  {e}")

    # reference: everything at 1
    w_all = {f"w_{x}": 1.0 for x in families}
    _, D_all, _ = _seqvec(seqs, multi, n_cl, **w_all,
                        **final_state.get("seqvec_inputs", {}))
    s_all = best_sil(D_all)
    print("  " + "-" * 40)
    print(f"  {'ALL (1,1..)':<12}{s_all:>+12.3f}  current reference")

    # combination of the 3 best families alone
    top3 = sorted(scores, key=scores.get, reverse=True)[:3]
    w_top = {f"w_{x}": (1.0 if x in top3 else 0.0) for x in families}
    _, D_top, _ = _seqvec(seqs, multi, n_cl, **w_top,
                        **final_state.get("seqvec_inputs", {}))
    s_top = best_sil(D_top)
    print(f"  {'top-3 only':<12}{s_top:>+12.3f}  ({', '.join(top3)})")

    print("=" * 58)
    print("\nHow to read :")
    print("  - 'noise' families (< 0.10) -> set their weight to 0")
    print("  - if 'top-3 only' > 'ALL' -> the discarded families were harmful")
    return scores, s_all, s_top


scores, s_all, s_top = diagnose_seqvec_families(final_state)

In [ ]:
# ===============================================================
#  CELL — Why these groups? (characteristics of each strategy)
#  Three complementary answers, for each group:
#   1. behaviour: median values (real units) + deviation from the mean (z score)
#   2. SeqVec families that separate the group from the others (score 0 -> 1)
#   3. frame states over-represented in the group
#  Groups: mapping_optimal if it exists, otherwise final_state (as the other cells).
# ===============================================================
import io
import contextlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kruskal
from sklearn.metrics import silhouette_samples

_EX_FAMILIES = ["hist", "temporal", "lastapp", "align", "dynamics", "duration", "speed", "path", "start"]
_EX_SHORT = {"distance_to_shrimp": "dist", "attack_side": "side", "bearing_to_prey": "bearing",
             "cuttle_approach_speed": "approach", "orbit_speed": "orbit", "who_moves": "who"}


def _ex_descriptors(df, ids, seqs):
    """Behaviour descriptors per individual, in real units (frames before the strike)."""
    d = df.copy()
    d["animal_id"] = d["animal_id"].astype(str)
    d = d[d["animal_id"].isin(ids)]
    if "time_to_strike_s" in d:
        d = d[d["time_to_strike_s"] <= 0]
    g = d.groupby("animal_id")
    cols = {}
    if "time_to_strike_s" in d:
        cols["duration (s)"] = -g["time_to_strike_s"].min()
    if "speed_cuttle_mm_s" in d:
        cols["distance travelled (mm)"] = g["speed_cuttle_mm_s"].sum() / 60
        cols["speed (mm/s)"] = g["speed_cuttle_mm_s"].mean()
        cols["% still"] = g["speed_cuttle_mm_s"].apply(lambda s: (s < 2).mean() * 100)
    if "distance_mm" in d:
        cols["start distance (mm)"] = g["distance_mm"].first()
        cols["mean distance (mm)"] = g["distance_mm"].median()
    if "cuttle_approach_speed_mm_s" in d:
        cols["approach (mm/s)"] = g["cuttle_approach_speed_mm_s"].mean()
    if "orbit_speed_deg_s" in d:
        cols["orbit (°/s)"] = g["orbit_speed_deg_s"].mean()
    if "who_moves" in d:
        cols["who_moves"] = g["who_moves"].mean()
    if "attack_side" in d:
        cols["side (°)"] = g["attack_side"].median()
        last = d[d["time_to_strike_s"] >= -1] if "time_to_strike_s" in d else d
        cols["side at strike (°)"] = last.groupby("animal_id")["attack_side"].median()
    if "bearing_to_prey" in d:
        cols["bearing (°)"] = g["bearing_to_prey"].median()
    out = pd.DataFrame(cols).reindex(ids)
    out["nb segments"] = [len(seqs[a]["seq"]) if a in seqs else np.nan for a in ids]
    return out


def _ex_state_labels(df, col, features):
    """Short description of each state: the 2 features that deviate most from the mean of the states."""
    feats = [f for f in features if f in df.columns]
    m = df.groupby(col)[feats].median()
    z = (m - m.mean()) / (m.std() + 1e-12)
    lab = {}
    for s, row in z.iterrows():
        top = row.abs().sort_values(ascending=False).index[:2]
        lab[s] = ", ".join(f"{_EX_SHORT.get(f, f[:8])} {'↑' if row[f] > 0 else '↓'}" for f in top)
    return lab


def explain_groups(final_state=None, df=None, groups=None, top=4, figure=True):
    """
    Explains the strategy groups: typical behaviour, SeqVec families
    that separate them, over-represented states, and a one-line summary per group.
    top: number of parameters cited in the summary of each group.
    """
    G  = globals()
    fs = G.get("final_state") if final_state is None else final_state
    df = G["df"] if df is None else df
    if groups is None:
        groups, src = ((G["mapping_optimal"], "mapping_optimal") if "mapping_optimal" in G
                       else (fs["mapping_letters"], "final_state"))
    else:
        src = "groups="
    groups = {str(a): g for a, g in groups.items() if g != "MONO"}
    ids    = [str(a) for a in fs["multi"] if str(a) in groups]
    lab    = np.array([groups[a] for a in ids])
    names  = sorted(set(lab))
    # ── Misclassified individuals : silhouette SeqVec de chaque individu ──
    pos = {str(a): i for i, a in enumerate(fs["multi"])}
    ix  = [pos[a] for a in ids]
    Dall = np.asarray(fs["D"])[np.ix_(ix, ix)]
    sil = silhouette_samples(Dall, lab, metric="precomputed") if len(names) > 1 else np.zeros(len(ids))
    closest = []
    for i in range(len(ids)):
        other = {g: Dall[i, lab == g].mean() for g in names if g != lab[i]}
        closest.append(min(other, key=other.get) if other else "")
    indiv = pd.DataFrame({"group": lab, "silhouette": sil, "closest group": closest}, index=ids)
    indiv["misclassified"] = indiv["silhouette"] < 0
    print(f"[GROUPS] groups : {src} | {len(ids)} individuals | {len(names)} groups : "
          + ", ".join(f"{g} ({(lab == g).sum()})" for g in names))

    # ── 1. Behaviour ──
    X = _ex_descriptors(df, ids, fs["seqs"])
    med = X.groupby(lab).median().loc[names]
    med.loc["ALL"] = X.median()
    med.insert(0, "n", list(pd.Series(lab).value_counts().loc[names]) + [len(ids)])
    n_mis = indiv.groupby("group")["misclassified"].sum().reindex(names).fillna(0).astype(int)
    med.insert(1, "misclassified", list(n_mis) + [int(indiv["misclassified"].sum())])
    med.insert(2, "silhouette", list(indiv.groupby("group")["silhouette"].mean().reindex(names)) + [sil.mean()])
    Z = (X - X.mean()) / (X.std() + 1e-12)
    zg = Z.groupby(lab).mean().loc[names]                 # mean deviation of the group from the mean of all, in standard deviations
    pval = {c: kruskal(*[X[c][lab == g].dropna() for g in names]).pvalue for c in X.columns}
    sig = [c for c in X.columns if pval[c] < 0.05]
    print("\n" + "═" * 100 + "\n1. BEHAVIOUR — medians per group (real units)\n" + "═" * 100)
    print(med.round(1).to_string())
    print("\nDeviation from the mean (z score: +1 = one standard deviation above the mean of all individuals):")
    print(zg.round(2).to_string())
    print("\nParameters that do NOT differ between groups (Kruskal-Wallis p >= 0.05) : " + (", ".join(c for c in X.columns if c not in sig) or "—"))

    # ── 2. SeqVec families ──
    fam_score = pd.DataFrame(index=names, dtype=float)
    W0 = {f"w_{f}": 0.0 for f in _EX_FAMILIES}
    inp = fs.get("seqvec_inputs", {})
    for fam in _EX_FAMILIES:
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                _, Df, _ = G["_seqvec"](fs["seqs"], ids, fs["n_cl"], **dict(W0, **{f"w_{fam}": 1.0}), **inp)
        except Exception:
            continue
        for g in names:
            m = lab == g
            within  = Df[np.ix_(m, m)][np.triu_indices(m.sum(), 1)].mean() if m.sum() > 1 else np.nan
            between = Df[np.ix_(m, ~m)].mean()
            fam_score.loc[g, fam] = (between - within) / (between + 1e-12)
    print("\n" + "═" * 100 + "\n2. SEQVEC FAMILIES — which ones separate each group?\n" + "═" * 100)
    print("Score = (distance to the others - internal distance) / distance to the others (0 = no separation, 1 = perfect):")
    print(fam_score.round(2).to_string())

    # ── 3. States ──
    col = fs.get("col", "cluster")
    st_lab = _ex_state_labels(df, col, list(G.get("FEATURES", [])))
    share = pd.DataFrame({g: pd.Series(np.concatenate([np.repeat(fs["seqs"][a]["seq"], fs["seqs"][a]["lengths"])
                                                       for a in np.array(ids)[lab == g]])).value_counts(normalize=True)
                          for g in names}).fillna(0).T
    overall = pd.Series(np.concatenate([np.repeat(fs["seqs"][a]["seq"], fs["seqs"][a]["lengths"]) for a in ids])
                        ).value_counts(normalize=True)
    print("\n" + "═" * 100 + "\n3. STATES — the most present in each group (and their description)\n" + "═" * 100)
    states_txt = {}
    for g in names:
        ratio = (share.loc[g] / overall.reindex(share.columns).replace(0, np.nan)).dropna()
        best = [s for s in share.loc[g].sort_values(ascending=False).index if share.loc[g, s] > 0.10][:3]
        over = [s for s in ratio.sort_values(ascending=False).index if ratio[s] > 1.5 and share.loc[g, s] > 0.05][:2]
        states_txt[g] = best
        print(f"  {g} : " + " | ".join(f"C{s} {share.loc[g, s]:.0%} ({st_lab.get(s, '')})" for s in best)
              + (f"   over-represented : " + ", ".join(f"C{s} x{ratio[s]:.1f}" for s in over) if over else ""))

    # ── 4. Misclassified individuals ──
    print("\n" + "═" * 100 + "\n4. MISCLASSIFIED INDIVIDUALS (silhouette < 0: closer to another group)\n" + "═" * 100)
    print("ID (silhouette → group it is closest to); sorted from the most to the least misclassified:")
    mis_txt = {}
    for g in names:
        m = indiv[(indiv["group"] == g) & indiv["misclassified"]].sort_values("silhouette")
        mis_txt[g] = ", ".join(f"{a} ({r['silhouette']:+.2f} → {r['closest group']})" for a, r in m.iterrows())
        print(f"  {g} ({len(m)}/{(lab == g).sum()}) : {mis_txt[g] or '—'}")

    # ── 5. Summary ──
    print("\n" + "═" * 100 + "\n5. SUMMARY — what characterises each group\n" + "═" * 100)
    summary = {}
    for g in names:
        zz = zg.loc[g, sig].dropna()
        key = zz.abs().sort_values(ascending=False).index[:top]
        parts = []
        for c in key:
            arrow = ("↑↑" if zz[c] > 1 else "↑") if zz[c] > 0 else ("↓↓" if zz[c] < -1 else "↓")
            parts.append(f"{c} {arrow} {med.loc[g, c]:.1f}")
        fams = fam_score.loc[g].dropna().sort_values(ascending=False).index[:2]
        summary[g] = " | ".join(parts)
        print(f"  {g} (n={(lab == g).sum()}, misclassified {n_mis[g]}) : {summary[g]}")
        print(f"        mostly separated by : {', '.join(fams)}   |   typical states : "
              + ", ".join(f"C{s} ({st_lab.get(s, '')})" for s in states_txt[g]))

    if figure:
        fig, axes = plt.subplots(1, 2, figsize=(18, 0.55 * len(names) + 3),
                                 gridspec_kw={"width_ratios": [len(X.columns), len(_EX_FAMILIES) * 0.8]})
        ax = axes[0]
        im = ax.imshow(zg.values, cmap="RdBu_r", vmin=-1.5, vmax=1.5, aspect="auto")
        ax.set_xticks(range(len(zg.columns)), zg.columns, rotation=40, ha="right", fontsize=8)
        ax.set_yticks(range(len(names)), [f"{g} (n={(lab == g).sum()})" for g in names])
        for i, g in enumerate(names):
            for j, c in enumerate(zg.columns):
                ax.text(j, i, f"{med.loc[g, c]:.0f}" if abs(med.loc[g, c]) >= 10 else f"{med.loc[g, c]:.1f}",
                        ha="center", va="center", fontsize=7,
                        color="white" if abs(zg.loc[g, c]) > 0.9 else "black",
                        weight="bold" if c in sig else "normal")
        ax.set_title("Behaviour: colour = deviation from the mean (z), text = median (bold = differs between groups)", fontsize=10)
        fig.colorbar(im, ax=ax, shrink=0.8, label="z score")
        ax = axes[1]
        im2 = ax.imshow(fam_score.values, cmap="Greens", vmin=0, vmax=max(0.3, np.nanmax(fam_score.values)),
                        aspect="auto")
        ax.set_xticks(range(len(fam_score.columns)), fam_score.columns, rotation=40, ha="right", fontsize=8)
        ax.set_yticks(range(len(names)), names)
        for i in range(len(names)):
            for j in range(len(fam_score.columns)):
                v = fam_score.values[i, j]
                ax.text(j, i, "—" if np.isnan(v) else f"{v:.2f}", ha="center", va="center", fontsize=7)
        ax.set_title("SeqVec families that separate each group", fontsize=10)
        fig.colorbar(im2, ax=ax, shrink=0.8)
        fig.tight_layout()
        plt.show()

    out_dir = G.get("OUT_DIR")
    if out_dir is not None:
        med.to_csv(out_dir / "groups_medians.csv")
        zg.to_csv(out_dir / "groups_zscores.csv")
        fam_score.to_csv(out_dir / "groups_families.csv")
        indiv.sort_values(["group", "silhouette"]).to_csv(out_dir / "groups_individuals.csv")
        print(f"\n[GROUPS] tables saved in {out_dir}")
    return {"medians": med, "zscores": zg, "families": fam_score, "states": share,
            "individuals": indiv, "misclassified": indiv[indiv["misclassified"]].sort_values(["group", "silhouette"]),
            "summary": summary, "pvalues": pval}


print("[OK] Group explanation defined — usage :  explication = explain_groups()")

In [ ]:
# Why these groups? (run after the explorer, and the regrouping if used)
explication = explain_groups(
    top = 4,          # number of parameters cited in the summary of each group
)

In [ ]:
# ===============================================================
#  CELL — Visualisation of the SeqVec groups + biological audit
#  Replays EXACTLY the _seqvec of show_strategy_explorer, projects
#  in 2D, colours by group, and measures whether the groups hold.
# ===============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_samples, silhouette_score
from scipy.spatial.distance import pdist, squareform


def visualize_strategy_groups(final_state, method="mds",
                              recompute_seqvec=None, annotate=True):
    """
    final_state : output of show_strategy_explorer.
    method      : "mds"  -> projects the clustering D matrix (most faithful)
                  "umap" -> UMAP on the SeqVec features
                  "pca"  -> PCA
    recompute_seqvec : dict of weights {w_hist:..} to REPLAY _seqvec with
                       other weightings and see the effect on the groups.
                       None = reuses the already computed D.
    """
    seqs  = final_state["seqs"]
    multi = final_state["multi"]
    strat = final_state["strat"]
    n_cl  = final_state["n_cl"]
    ml    = final_state.get("mapping_letters", {})

    ids = [str(a) for a in multi]
    grp = np.array([ml.get(a, str(strat.get(a, "?"))) for a in ids])

    # ── Distance: existing D, or recomputed with other weights ─
    if recompute_seqvec is not None:
        # replays _seqvec (must be importable in the notebook)
        _, D, _ = _seqvec(seqs, multi, n_cl, **recompute_seqvec,
                        **final_state.get("seqvec_inputs", {}))
        title_w = "  ·  modified weights"
    else:
        D = np.asarray(final_state["D"])
        title_w = ""

    # ── 2D projection ──────────────────────────────────────────
    if method == "mds":
        from sklearn.manifold import MDS
        Z = MDS(n_components=2, dissimilarity="precomputed",
                random_state=42, normalized_stress="auto").fit_transform(D)
        tm = "MDS on D (actual clustering distance)"
        Xsil = D                     # silhouette on the precomputed distance
        sil_metric = "precomputed"
    elif method == "umap":
        import umap
        # starting from the features: rebuilding X from D is impossible,
        # so D is used as input with metric='precomputed'
        Z = umap.UMAP(n_components=2, metric="precomputed", random_state=42,
                      n_neighbors=min(15, len(D)-1)).fit_transform(D)
        tm = "UMAP on D"; Xsil = D; sil_metric = "precomputed"
    else:
        from sklearn.decomposition import PCA
        # PCA needs coordinates: classical MDS first
        from sklearn.manifold import MDS
        coords = MDS(n_components=min(10, len(D)-1), dissimilarity="precomputed",
                     random_state=42, normalized_stress="auto").fit_transform(D)
        Z = PCA(n_components=2).fit_transform(coords)
        tm = "PCA (via 10D MDS)"; Xsil = D; sil_metric = "precomputed"

    # ── Silhouette (do the groups hold?) ────────────────
    lab_num = pd.factorize(grp)[0]
    counts = pd.Series(lab_num).value_counts()
    valid = np.isin(lab_num, counts[counts >= 2].index)
    sil_ind = np.full(len(lab_num), np.nan)
    sil_glob = np.nan
    if valid.sum() >= 3 and len(set(lab_num[valid])) >= 2:
        sil_ind[valid] = silhouette_samples(Xsil[np.ix_(valid, valid)],
                                             lab_num[valid], metric=sil_metric)
        sil_glob = silhouette_score(Xsil[np.ix_(valid, valid)],
                                    lab_num[valid], metric=sil_metric)

    # ── Colours ───────────────────────────────────────────────
    gs = sorted(set(grp))
    pal = {g: STRATEGY_PALETTE[i % len(STRATEGY_PALETTE)] for i, g in enumerate(gs)}
    cols = [pal[g] for g in grp]

    # ══ FIGURE ═════════════════════════════════════════════════
    fig = plt.figure(figsize=(17, 6))
    fig.patch.set_facecolor("#1a1a2e")

    # P1 — coloured projection
    ax = fig.add_subplot(1, 3, 1); ax.set_facecolor("#0d0d1a")
    ax.scatter(Z[:, 0], Z[:, 1], c=cols, s=110, edgecolor="white",
               linewidth=0.6, zorder=3)
    if annotate:
        for (x, y), a in zip(Z, ids):
            ax.text(x, y, a, fontsize=6, color="#eee", ha="center", va="center")
    for g in gs:
        m = grp == g
        ax.scatter(Z[m, 0].mean(), Z[m, 1].mean(), marker="X", s=220,
                   c=pal[g], edgecolor="black", linewidth=1.5, zorder=5)
    ax.set_title(f"{tm}{title_w}", color="#ccc", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values(): sp.set_edgecolor("#333")
    ax.legend(handles=[mpatches.Patch(color=pal[g],
              label=f"{g} (n={int((grp==g).sum())})") for g in gs],
              fontsize=7, framealpha=0.3)

    # P2 — silhouette per individual, sorted by group
    ax2 = fig.add_subplot(1, 3, 2); ax2.set_facecolor("#0d0d1a")
    order = sorted(range(len(ids)),
                   key=lambda i: (grp[i], -(sil_ind[i] if not np.isnan(sil_ind[i]) else -9)))
    ax2.barh(range(len(order)), sil_ind[order],
             color=[cols[i] for i in order])
    ax2.axvline(0, color="#888", lw=1)
    if not np.isnan(sil_glob):
        ax2.axvline(sil_glob, color="#FFD700", lw=1.5, ls="--",
                    label=f"mean {sil_glob:.2f}")
        ax2.legend(fontsize=8, framealpha=0.3)
    ax2.set_yticks([]); ax2.set_xlim(-0.6, 1)
    ax2.set_xlabel("silhouette (negative = misclassified)", color="#aaa", fontsize=8)
    ax2.set_title("Cohesion per individual", color="#ccc", fontsize=9)
    ax2.tick_params(colors="#aaa")
    for sp in ax2.spines.values(): sp.set_edgecolor("#333")

    # P3 — time-per-cluster profile of each group
    ax3 = fig.add_subplot(1, 3, 3); ax3.set_facecolor("#0d0d1a")
    for g in gs:
        profs = []
        for a in np.array(ids)[grp == g]:
            sq, lg = seqs[a]["seq"], seqs[a]["lengths"]
            t = sum(lg)
            if t == 0: continue
            h = np.zeros(n_cl)
            for c, l in zip(sq, lg):
                if c < n_cl: h[c] += l / t
            profs.append(h)
        if profs:
            ax3.plot(range(n_cl), np.mean(profs, 0), "-o", color=pal[g],
                     label=g, lw=2, markersize=4)
    ax3.set_xticks(range(n_cl))
    ax3.set_xticklabels([f"C{c}" for c in range(n_cl)], fontsize=7, color="#aaa")
    ax3.set_ylabel("proportion of time", color="#aaa", fontsize=8)
    ax3.set_title("Mean profile per group", color="#ccc", fontsize=9)
    ax3.tick_params(colors="#aaa"); ax3.legend(fontsize=7, framealpha=0.3)
    for sp in ax3.spines.values(): sp.set_edgecolor("#333")

    plt.tight_layout(); plt.show()

    # ── Summary ─────────────────────────────────────────────────
    print(f"\n{'='*56}")
    print(f"QUALITY — {len(gs)} groups | global silhouette {sil_glob:+.3f} "
          f"({'solid' if sil_glob>0.25 else 'weak' if sil_glob>0 else 'inconsistent'})")
    print(f"{'='*56}")
    print(f"  {'group':<8}{'n':>4}{'silhouette':>12}  misclassified")
    for g in gs:
        m = grp == g
        s = sil_ind[m]; s = s[~np.isnan(s)]
        neg = int((s < 0).sum())
        print(f"  {g:<8}{int(m.sum()):>4}{(np.mean(s) if len(s) else 0):>+12.3f}"
              f"     {neg}/{int(m.sum())}")
    bad = sorted([(ids[i], grp[i], sil_ind[i]) for i in range(len(ids))
                  if not np.isnan(sil_ind[i]) and sil_ind[i] < 0],
                 key=lambda x: x[2])
    if bad:
        print(f"\n  Misclassified individuals (candidates for reclassifying/merging) :")
        for a, g, s in bad[:10]:
            print(f"    animal {a:<6} group {g:<4} sil={s:+.3f}")
    print("=" * 56)
    return Z, grp, sil_ind


# ── Usage ────────────────────────────────────────────────
# Z_mds, grp, sil = visualize_strategy_groups(final_state, method="mds")
#
# Test other weights WITHOUT rerunning the explorer:
# Z_mds, grp, sil = visualize_strategy_groups(final_state, method="mds",
#     recompute_seqvec=dict(w_hist=2.0, w_temporal=1.0, w_lastapp=1.0,
#                           w_duration=0.5, w_align=1.0, w_dynamics=1.0, w_speed=0.5,
#                           w_path=0.5, w_start=0.5))

In [ ]:
Z_mds, grp, sil = visualize_strategy_groups(final_state, method="mds")

## Medoids (cluster, then supercluster)

In [ ]:
#def plot_strategy_profiles

from scipy.spatial.distance import cdist
import matplotlib.colors as mcolors


def plot_strategy_profiles(mapping_optimal, final_state, df, features=FEATURES,
                           cmap_cl=None, norm_cl=None):

    # Default values if not provided
    n_cl_local = final_state["n_cl"]
    if cmap_cl is None:
        cmap_cl = CMAP_CLUSTER
    if norm_cl is None:
        norm_cl = NORM_CLUSTER

    seqs = final_state["seqs"]
    n_cl = final_state["n_cl"]

    df_w = df.copy()
    df_w["animal_id"] = df_w["animal_id"].astype(str)

    unique_groups = sorted(set(v for v in mapping_optimal.values() if v != "MONO"))
    num_to_letter = {g: chr(65+i) for i, g in enumerate(unique_groups)}
    num_to_letter["MONO"] = "MONO"
    mapping_letters  = {aid: num_to_letter[g] for aid, g in mapping_optimal.items()}
    strategy_letters = sorted(set(v for v in mapping_letters.values() if v != "MONO"))
    n_strat          = len(strategy_letters)

    pal         = {s: STRATEGY_PALETTE[i % len(STRATEGY_PALETTE)]
                   for i, s in enumerate(strategy_letters)}
    pal["MONO"] = "#666666"

    # ── DTW for the medoid ──────────────────────────────────────
    def _dtw(seq_a, seq_b, n_cl):
        if not seq_a or not seq_b: return np.inf
        centers = np.eye(n_cl)
        tA = np.array([centers[min(c, n_cl-1)] for c in seq_a])
        tB = np.array([centers[min(c, n_cl-1)] for c in seq_b])
        TA, TB = len(tA), len(tB)
        cost   = cdist(tA, tB, metric='euclidean')
        D      = np.full((TA, TB), np.inf)
        D[0,0] = cost[0,0]
        for i in range(1, TA): D[i,0] = D[i-1,0] + cost[i,0]
        for j in range(1, TB): D[0,j] = D[0,j-1] + cost[0,j]
        for i in range(1, TA):
            for j in range(1, TB):
                D[i,j] = cost[i,j] + min(D[i-1,j], D[i,j-1], D[i-1,j-1])
        return D[TA-1, TB-1]

    def _medoid(members):
        n = len(members)
        if n == 1: return members[0], seqs[members[0]]["seq"]
        dists = np.zeros(n)
        for i in range(n):
            for j in range(n):
                if i != j:
                    dists[i] += _dtw(seqs[members[i]]["seq"],
                                     seqs[members[j]]["seq"], n_cl)
        best = int(np.argmin(dists))
        return members[best], seqs[members[best]]["seq"]

    angles   = np.linspace(0, 2*np.pi, n_cl, endpoint=False)
    ang_full = np.concatenate([angles, [angles[0]]])

    profiles = {}
    for s in strategy_letters:
        members         = [aid for aid, g in mapping_letters.items() if g == s]
        members_in_seqs = [m for m in members if m in seqs]
        if not members_in_seqs: continue

        hists = []
        for aid in members_in_seqs:
            sq, lg = seqs[aid]["seq"], seqs[aid]["lengths"]
            total  = sum(lg)
            if total == 0: continue
            h = np.zeros(n_cl)
            for c, l in zip(sq, lg):
                if c < n_cl: h[c] += l / total
            hists.append(h)
        mean_hist = np.mean(hists, axis=0) if hists else np.zeros(n_cl)

        print(f"  [MEDOID] Strategy {s} ({len(members_in_seqs)} ind)...", end=" ")
        medoid_aid, modal_compressed = _medoid(members_in_seqs)
        print(f"-> ID {medoid_aid}")

        durations = {c: [] for c in range(n_cl)}
        for aid in members_in_seqs:
            for c, l in zip(seqs[aid]["seq"], seqs[aid]["lengths"]):
                if c < n_cl: durations[c].append(l)
        median_durations = {c: np.median(durations[c]) if durations[c] else 0
                         for c in range(n_cl)}

        trans_count = {}
        for aid in members_in_seqs:
            sq = seqs[aid]["seq"]
            for a, b in zip(sq[:-1], sq[1:]):
                trans_count[(a,b)] = trans_count.get((a,b), 0) + 1
        top_trans = sorted(trans_count.items(), key=lambda x: -x[1])[:3]

        feat_means = {}
        if features:
            ids_str = [str(m) for m in members_in_seqs]
            mask    = df_w["animal_id"].isin(ids_str)
            if mask.any():
                feat_means = df_w[mask][features].mean().to_dict()

        profiles[s] = {
            "members"         : members_in_seqs,
            "all_members"     : members,
            "n"               : len(members_in_seqs),
            "mean_hist"       : mean_hist,
            "modal_seq"       : modal_compressed,
            "medoid_aid"      : medoid_aid,
            "median_durations"   : median_durations,
            "top_transitions" : top_trans,
            "dominant_cluster": int(np.argmax(mean_hist)),
            "feat_means"      : feat_means,
            "mean_seq_len"    : np.mean([len(seqs[a]["seq"])
                                         for a in members_in_seqs]),
            "mean_n_clusters" : np.mean([len(set(seqs[a]["seq"]))
                                         for a in members_in_seqs]),
        }
    print(f"[DEBUG] mapping_optimal sample : {list(mapping_optimal.items())[:5]}")
    print(f"[DEBUG] mapping_letters sample : {list(mapping_letters.items())[:5]}")
    print(f"[DEBUG] strategy_letters : {strategy_letters}")
    print(f"[DEBUG] seqs keys sample : {list(seqs.keys())[:5]}")

    for s in strategy_letters:
        members = [aid for aid, g in mapping_letters.items() if g == s]
        in_seqs = [m for m in members if m in seqs]
        print(f"  {s} : {len(members)} members, {len(in_seqs)} in seqs")
    # ════════════════════════════════════════════════════════
    # FIGURE 1 — Radars
    # ════════════════════════════════════════════════════════
    n_cols = min(4, n_strat)
    n_rows = (n_strat + n_cols - 1) // n_cols
    fig1   = plt.figure(figsize=(n_cols * 5, n_rows * 5))
    fig1.patch.set_facecolor("#1a1a2e")
    fig1.suptitle("Behavioural profiles of the strategies",
                  color="#cccccc", fontsize=13, y=1.01)

    for si, s in enumerate(strategy_letters):
        if s not in profiles: continue
        p   = profiles[s]
        col = pal[s]
        ax  = fig1.add_subplot(n_rows, n_cols, si+1, polar=True)
        ax.set_facecolor("#0d0d1a")
        vals_f = np.concatenate([p["mean_hist"], [p["mean_hist"][0]]])
        ax.plot(ang_full, vals_f, lw=2.5, color=col)
        ax.fill(ang_full, vals_f, alpha=0.3, color=col)
        ax.set_xticks(angles)
        ax.set_xticklabels([f"C{c}" for c in range(n_cl)],
                            fontsize=7, color="#aaaaaa")
        ax.set_ylim(0, max(p["mean_hist"].max()*1.2, 0.01))
        ax.set_yticks([])
        ax.grid(color="#333333", lw=0.5)
        ax.spines["polar"].set_color("#333333")
        modal_str = "→".join([f"C{c}" for c in p["modal_seq"][:6]])
        if len(p["modal_seq"]) > 6: modal_str += "…"
        ax.set_title(
            f"Strategy {s}  (n={p['n']})\n"
            f"Dominant : C{p['dominant_cluster']}\n"
            f"Medoid : ID {p['medoid_aid']}\n"
            f"{modal_str}",
            color=col, fontsize=9, pad=12
        )
    plt.tight_layout(); plt.show()

    # ════════════════════════════════════════════════════════
    # FIGURE 2 — MEDOID sequence (1 real individual)
    # Visual improvement: merged blocks, thickness by duration
    # ════════════════════════════════════════════════════════
    fig2, axes = plt.subplots(n_strat, 1,
                               figsize=(14, max(3, n_strat * 1.6)))
    fig2.patch.set_facecolor("#1a1a2e")
    fig2.suptitle(
        "Medoid sequence per strategy\n"
        "(most representative real individual — DTW medoid)\n"
        "width ∝ duration | consecutive identical blocks merged",
        color="#cccccc", fontsize=11
    )
    if n_strat == 1: axes = [axes]

    for si, (s, ax) in enumerate(zip(strategy_letters, axes)):
        if s not in profiles: continue
        p    = profiles[s]
        col  = pal[s]

        # Sequence of the medoid (real individual)
        seq_raw  = seqs[p["medoid_aid"]]["seq"]
        lgts_raw = seqs[p["medoid_aid"]]["lengths"]

        # ── Merge consecutive identical blocks ───────────
        # Avoids the visual repetition C0 C0 C0 → a single large C0
        seq_fused, lgts_fused = [], []
        for c, l in zip(seq_raw, lgts_raw):
            if seq_fused and seq_fused[-1] == c:
                lgts_fused[-1] += l
            else:
                seq_fused.append(c)
                lgts_fused.append(l)

        ax.set_facecolor("#0d0d1a")
        for sp in ax.spines.values(): sp.set_edgecolor("#333333")
        ax.set_yticks([])
        ax.tick_params(colors="#555555", labelsize=7)

        x    = 0
        dmax = max(lgts_fused) if lgts_fused else 1
        for c, dur in zip(seq_fused, lgts_fused):
            c_color = mcolors.to_hex(cmap_cl(norm_cl(c)))

            # Height proportional to duration → important blocks = thicker
            h = 0.35 + 0.55 * (dur / dmax)

            ax.barh(0, dur, left=x, height=h,
                    color=c_color, edgecolor="#0d0d1a", lw=0.8,
                    alpha=0.9)

            # Label only if the block is wide enough
            if dur >= dmax * 0.04:
                fontsize = min(9, max(5, int(7 * dur / dmax * 3)))
                ax.text(x + dur/2, 0, f"C{c}",
                        ha="center", va="center",
                        fontsize=fontsize, color="white",
                        fontweight="bold")
            x += dur

        ax.set_xlim(0, x)
        ax.set_ylim(-0.55, 0.55)

        # Group colour bar on the left
        ax.barh(0, x * 0.008, left=-x*0.015, height=1.1,
                color=col, alpha=0.9)

        total_frames = sum(lgts_fused)
        n_blocks_orig = len(seq_raw)
        n_blocks_fused  = len(seq_fused)

        trans_str = "  |  ".join(
            [f"C{a}→C{b} ({n}×)" for (a,b), n in p["top_transitions"]]
        )

        ax.set_ylabel(
            f"Strat. {s}\nn={p['n']}\nID {p['medoid_aid']}",
            color=col, fontsize=8, rotation=0,
            ha="right", va="center", labelpad=60
        )
        ax.text(x * 1.005, 0.25,
                f"{total_frames}f",
                va="center", ha="left", fontsize=6, color="#666666")
        ax.text(x * 1.005, -0.25,
                f"{n_blocks_fused} blocks ({n_blocks_orig} raw)",
                va="center", ha="left", fontsize=6, color="#555555")
        ax.text(x * 1.005, 0.0,
                f"↑ {trans_str}",
                va="center", ha="left", fontsize=6, color="#777777")

    plt.tight_layout(); plt.show()

    # ════════════════════════════════════════════════════════
    # FIGURE 3 — Summary table + list of individuals
    # ════════════════════════════════════════════════════════
    fig3, ax = plt.subplots(figsize=(18, max(3, n_strat * 0.9 + 1.5)))
    fig3.patch.set_facecolor("#1a1a2e")
    ax.set_facecolor("#0d0d1a"); ax.axis("off")

    headers = (["Strategy", "N", "Medoid", "Dominant cluster",
                "Mean blocks", "Distinct clusters", "Top transition",
                "Individuals (IDs)"]
               + [f.replace("_"," ")[:10] for f in features])

    table_data = []
    for s in strategy_letters:
        if s not in profiles: continue
        p = profiles[s]

        top_t = (f"C{p['top_transitions'][0][0][0]}→"
                 f"C{p['top_transitions'][0][0][1]}"
                 if p["top_transitions"] else "—")

        # List of individuals — sorted, comma-separated
        ids_str = ", ".join(sorted([str(m) for m in p["members"]]))
        # Truncate if too long for the table
        if len(ids_str) > 60:
            ids_str = ids_str[:57] + "…"

        row = [
            f"  {s}  ",
            str(p["n"]),
            str(p["medoid_aid"]),
            f"C{p['dominant_cluster']}  ({p['mean_hist'][p['dominant_cluster']]:.0%})",
            f"{p['mean_seq_len']:.1f}",
            f"{p['mean_n_clusters']:.1f}",
            top_t,
            ids_str,
        ]
        for feat in features:
            v = p["feat_means"].get(feat, np.nan)
            row.append(f"{v:.2f}" if not np.isnan(v) else "—")
        table_data.append(row)

    tbl = ax.table(cellText=table_data, colLabels=headers,
                   loc="center", cellLoc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(7)
    tbl.scale(1, 2.2)

    for (row, col_idx), cell in tbl.get_celld().items():
        cell.set_edgecolor("#333333")
        if row == 0:
            cell.set_facecolor("#1a1a3e")
            cell.set_text_props(color="white", fontweight="bold")
        else:
            s = strategy_letters[row-1] if row-1 < len(strategy_letters) else None
            cell.set_facecolor("#0d0d1a")
            cell.set_text_props(color="#cccccc")
            if col_idx == 0 and s:
                cell.set_facecolor(pal.get(s, "#0d0d1a") + "55")
                cell.set_text_props(color=pal.get(s, "white"),
                                    fontweight="bold")
            # Individuals column: lightly coloured
            if col_idx == 7 and s:
                cell.set_facecolor(pal.get(s, "#0d0d1a") + "22")
                cell.set_text_props(color="#aaaaaa", fontsize=6)

    ax.set_title("Summary table — DTW medoid + list of individuals",
                 color="#cccccc", fontsize=11, pad=15)
    plt.tight_layout(); plt.show()

    # ════════════════════════════════════════════════════════
    # FIGURE 4 — Detailed list of individuals per group
    # ════════════════════════════════════════════════════════
    fig4, ax4 = plt.subplots(figsize=(14, max(4, n_strat * 1.2 + 1)))
    fig4.patch.set_facecolor("#1a1a2e")
    ax4.set_facecolor("#0d0d1a"); ax4.axis("off")

    ax4.set_title("Individuals per strategy",
                  color="#cccccc", fontsize=12, pad=10)

    y_pos = 0.97
    step  = 1.0 / (n_strat + 1)

    for s in strategy_letters:
        if s not in profiles: continue
        p   = profiles[s]
        col = pal[s]

        ids_sorted = sorted([str(m) for m in p["members"]])
        ids_chunks = [ids_sorted[i:i+20] for i in range(0, len(ids_sorted), 20)]

        # Group label
        ax4.text(0.01, y_pos,
                 f"Strategy {s}  (n={p['n']})  —  medoid : {p['medoid_aid']}",
                 transform=ax4.transAxes,
                 color=col, fontsize=10, fontweight='bold', va='top')

        # IDs on one or more lines
        for li, chunk in enumerate(ids_chunks):
            ax4.text(0.12, y_pos - 0.015 * (li + 1),
                     "  ".join(chunk),
                     transform=ax4.transAxes,
                     color="#cccccc", fontsize=7, va='top',
                     fontfamily='monospace')

        y_pos -= step

    plt.tight_layout(); plt.show()

    # ── Terminal summary ───────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"PROFILES ({n_strat} groups) — DTW medoids")
    print(f"{'='*60}")
    for s in strategy_letters:
        if s not in profiles: continue
        p = profiles[s]
        ids_sorted = sorted([str(m) for m in p["members"]])
        print(f"\nStrategy {s}  (n={p['n']})  —  medoid : ID {p['medoid_aid']}")
        print(f"  Individuals : {', '.join(ids_sorted)}")
        print(f"  Dominant  : C{p['dominant_cluster']} "
              f"({p['mean_hist'][p['dominant_cluster']]:.0%})")
        print(f"  Mean blocks : {p['mean_seq_len']:.1f} | "
              f"Distinct clusters : {p['mean_n_clusters']:.1f}")
        if p["top_transitions"]:
            print(f"  Top trans : " +
                  ", ".join([f"C{a}→C{b}({n}×)"
                             for (a,b),n in p["top_transitions"]]))

    return profiles, mapping_letters

In [ ]:
# Cell 19 — must be run BEFORE cell 30
profiles, mapping_letters = plot_strategy_profiles(
    mapping_optimal,
    final_state,
    df,
    features = FEATURES,
)

In [ ]:
# ── Same strategy groups, sequences rebuilt on SUPERCLUSTERS ──
_run = final_state.get("min_run_len", final_state.get("seuil_min", 15))
_df = df.copy(); _df["animal_id"] = _df["animal_id"].astype(str)
seqs_super, _, _ = _extract_sequences(_df, "supercluster", _run)
final_state_super = dict(final_state, seqs=seqs_super, col="supercluster",
                         n_cl=int(df["supercluster"].nunique()))
profiles_super, _ = plot_strategy_profiles(
    mapping_optimal if "mapping_optimal" in globals() else final_state["mapping_letters"],
    final_state_super, df, features=FEATURES, cmap_cl=CMAP_SUPER, norm_cl=NORM_SUPER)

## State probabilities over time (cluster, then supercluster)

In [ ]:
#def plot_cluster_probability_over_time

import matplotlib as mpl

def plot_cluster_probability_over_time(profiles, seqs, n_cl,
                                        strategy_letters, pal,cmap_cl, norm_cl,
                                        n_bins=100, smooth_sigma=3,
                                        min_prob_threshold=0.05):
    """
    Two figures per run:

    Figure 1 — all clusters (complete stacked area chart)
    Figure 2 — filtered clusters: in each bin, the clusters whose
               probability < min_prob_threshold are set to 0
               then renormalised → shows only the clusters that are
               significant at each point in time

    min_prob_threshold : minimum probability threshold (default 0.05 = 5%)
                         clusters below it are hidden in fig 2
    """
    from scipy.ndimage import gaussian_filter1d

    n_strat = len(strategy_letters)
    t_axis  = np.linspace(0, 1, n_bins)

    # ── Probabilities for all strategies ────
    # Computed once, drawn as two figures
    all_probs = {}   # {strategy: array (n_cl, n_bins)}

    for s in strategy_letters:
        if s not in profiles: continue
        p       = profiles[s]
        members = [m for m in p["members"] if m in seqs]
        if not members: continue

        cluster_matrix = np.full((len(members), n_bins), -1, dtype=int)

        for mi, aid in enumerate(members):
            seq     = seqs[aid]["seq"]
            lengths = seqs[aid]["lengths"]
            total   = sum(lengths)
            if total == 0: continue

            # Frame-by-frame timeline from the blocks
            timeline = []
            for c, l in zip(seq, lengths):
                timeline.extend([c] * int(l))
            timeline = np.array(timeline)
            n_frames = len(timeline)

            for b in range(n_bins):
                i_start = int(b * n_frames / n_bins)
                i_end   = int((b + 1) * n_frames / n_bins)
                chunk   = timeline[i_start:i_end]
                if len(chunk) > 0:
                    cluster_matrix[mi, b] = int(
                        np.bincount(chunk, minlength=n_cl).argmax()
                    )

        # Raw probabilities
        prob = np.zeros((n_cl, n_bins))
        for b in range(n_bins):
            col_b = cluster_matrix[:, b]
            valid = col_b[col_b >= 0]
            if len(valid) == 0: continue
            for c in range(n_cl):
                prob[c, b] = np.sum(valid == c) / len(valid)

        # Smoothing
        if smooth_sigma > 0:
            for c in range(n_cl):
                prob[c] = gaussian_filter1d(prob[c], sigma=smooth_sigma,
                                             mode='nearest')
            col_sums = prob.sum(axis=0)
            col_sums[col_sums == 0] = 1
            prob /= col_sums

        all_probs[s] = prob

    # ══════════════════════════════════════════════════════════
    # Drawing function shared by both figures
    # ══════════════════════════════════════════════════════════
    def _draw_prob_fig(prob_dict, title_suffix, filter_label=""):

        fig, axes = plt.subplots(n_strat, 1,
                                  figsize=(14, max(3, n_strat * 2.2)))
        fig.patch.set_facecolor("#1a1a2e")
        fig.suptitle(
            f"Probability of being in each cluster over time\n"
            f"{title_suffix}",
            color="#cccccc", fontsize=12
        )
        if n_strat == 1: axes_list = [axes]
        else:            axes_list = list(axes)

        for si, (s, ax) in enumerate(zip(strategy_letters, axes_list)):
            if s not in profiles or s not in prob_dict: continue
            p   = profiles[s]
            col = pal[s]

            members = [m for m in p["members"] if m in seqs]
            prob    = prob_dict[s]

            # Stacked area chart
            ax.set_facecolor("#0d0d1a")
            for sp in ax.spines.values(): sp.set_edgecolor("#333333")
            ax.tick_params(colors="#aaaaaa", labelsize=8)

            bottom = np.zeros(n_bins)
            for c in range(n_cl):
                c_color = cmap_cl(norm_cl(c))
                vals    = prob[c]

                ax.fill_between(t_axis, bottom, bottom + vals,
                                color=c_color, alpha=0.85,
                                label=f"C{c}", zorder=2)
                ax.plot(t_axis, bottom + vals,
                        color="#0d0d1a", lw=0.4, alpha=0.6, zorder=3)
                bottom += vals

            # Midline
            ax.axvline(0.5, color="#ffffff", lw=0.8, ls=":",
                       alpha=0.3, zorder=4)

            # Dominant cluster annotations
            dominant  = np.argmax(prob, axis=0)
            prev_c    = dominant[0]
            run_start = 0
            for b in range(1, n_bins):
                if dominant[b] != prev_c or b == n_bins - 1:
                    run_len = b - run_start
                    if run_len >= n_bins * 0.05:
                        mid_b = (run_start + b) // 2
                        mid_x = t_axis[mid_b]
                        bot_c = prob[:prev_c, mid_b].sum()
                        top_c = bot_c + prob[prev_c, mid_b]
                        mid_y = (bot_c + top_c) / 2
                        if prob[prev_c, mid_b] > 0.08:
                            ax.text(mid_x, mid_y, f"C{prev_c}",
                                    ha="center", va="center",
                                    fontsize=7, color="white",
                                    fontweight="bold", zorder=5,
                                    alpha=0.9)
                    run_start = b
                    prev_c    = dominant[b]

            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
            ax.set_ylabel(
                f"Strat. {s}\nn={len(members)}",
                color=col, fontsize=9, rotation=0,
                ha="right", va="center", labelpad=55
            )
            ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
            ax.set_yticklabels(["0%","25%","50%","75%","100%"],
                                color="#aaaaaa", fontsize=7)
            ax.grid(axis="x", color="#222233", alpha=0.5, lw=0.5)

            if si == n_strat - 1:
                ax.set_xlabel(
                    "Normalised time position  (0=start  1=end)",
                    color="#aaaaaa", fontsize=10)
            else:
                ax.set_xticklabels([])

            # Strategy colour bar on the left
            ax.barh(0.5, 0.008, left=-0.02, height=1.0,
                    color=col, alpha=0.9, transform=ax.transData,
                    clip_on=False, zorder=5)

            # Threshold annotation in the filtered figure
            if filter_label:
                ax.text(0.01, 0.97,
                        f"threshold = {int(min_prob_threshold*100)}%",
                        transform=ax.transAxes,
                        color="#777788", fontsize=7,
                        va="top", ha="left")

        # Legend
        legend_patches = [
            mpatches.Patch(color=cmap_cl(norm_cl(c)), label=f"C{c}")
            for c in range(n_cl)
        ]
        leg = axes_list[0].legend(
            handles=legend_patches,
            loc="upper right", ncol=min(n_cl, 6),
            facecolor="#0d0d1a", labelcolor="#cccccc",
            fontsize=8, title="Clusters", title_fontsize=8,
        )
        leg.get_frame().set_edgecolor("#444444")
        leg.get_title().set_color("#aaaaaa")

        plt.tight_layout()
        plt.show()

    # ══════════════════════════════════════════════════════════
    # FIGURE 1 — All clusters
    # ══════════════════════════════════════════════════════════
    _draw_prob_fig(
        prob_dict    = all_probs,
        title_suffix = "(all clusters — normalised time position)"
    )

    # ══════════════════════════════════════════════════════════
    # FIGURE 2 — Filtered clusters (prob < threshold → hidden)
    # In each bin: the clusters whose prob < min_prob_threshold
    # are set to 0, then the remaining probabilities are renormalised
    # → only the "active" clusters at each moment are shown
    # ══════════════════════════════════════════════════════════
    all_probs_filtered = {}

    for s, prob in all_probs.items():
        prob_f = prob.copy()

        # Mask: set clusters below the threshold to 0, bin by bin
        prob_f[prob_f < min_prob_threshold] = 0

        # Renormalise bin by bin so the sum stays = 1
        # → the freed space is redistributed proportionally
        col_sums = prob_f.sum(axis=0)
        col_sums[col_sums == 0] = 1
        prob_f /= col_sums

        all_probs_filtered[s] = prob_f

    _draw_prob_fig(
        prob_dict    = all_probs_filtered,
        title_suffix = (f"filtered clusters — threshold {int(min_prob_threshold*100)}% "
                        f"(clusters < {int(min_prob_threshold*100)}% hidden "
                        f"and redistributed)"),
        filter_label = True
    )


# ── Call ─────────────────────────────────────────────────────
# Prerequisite: profiles comes from plot_strategy_profiles
# profiles, mapping_letters = plot_strategy_profiles(...)

seqs             = final_state["seqs"]
n_cl             = final_state["n_cl"]
strategy_letters = sorted(profiles.keys())   # A, B, C... from profiles

pal         = {s: STRATEGY_PALETTE[i % len(STRATEGY_PALETTE)]
               for i, s in enumerate(strategy_letters)}
pal["MONO"] = "#666666"

# Cluster colours — same palette as everywhere else
cmap_cl = mpl.colors.ListedColormap(PALETTE)
norm_cl = mpl.colors.BoundaryNorm(range(len(PALETTE) + 1),
                                   ncolors=len(PALETTE))

In [ ]:
plot_cluster_probability_over_time(
    profiles          = profiles,
    seqs              = seqs,
    n_cl              = n_cl,
    strategy_letters  = strategy_letters,
    pal               = pal,
    cmap_cl           = cmap_cl,
    norm_cl           = norm_cl,
    n_bins            = 100,
    smooth_sigma      = 3,
    min_prob_threshold= 0.05,
)

In [ ]:
# ── State probabilities over time, SUPERCLUSTER version ──
plot_cluster_probability_over_time(
    profiles           = profiles_super,
    seqs               = final_state_super["seqs"],
    n_cl               = final_state_super["n_cl"],
    strategy_letters   = sorted(profiles_super.keys()),
    pal                = pal,
    cmap_cl            = CMAP_SUPER,
    norm_cl            = NORM_SUPER,
    n_bins             = 100,
    smooth_sigma       = 3,
    min_prob_threshold = 0.05,
)